# PaddleOCR-VL-1.6 — offline 4×L4 BF16 vs GPTQ W8A16/W4A16 speed test

Attach the saved output of the dependency-builder notebook, select **4× NVIDIA L4**, set **Internet OFF**, and run all cells.

The attached bundle was prepared earlier with Internet ON and no accelerator. This notebook installs only from that bundle, quantizes the frozen BF16 source into W8A16-G128 and W4A16-G128 on physical GPU 0, audits both checkpoints, and then benchmarks all three variants. No network access is used.

The PaddleOCR pipeline, layout model, generation parameters, test pages, warmups, calibration examples, target matrices, and BF16 activation dtype stay fixed. Low-bit variants are accepted only if vLLM selects its native Marlin WNA16 kernel.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import math
import os
import re
import shutil
import statistics
import subprocess
import sys
import time
import urllib.request
import zipfile

OUTPUT = Path('/kaggle/working/l4_wna16_benchmark_results')
if OUTPUT.exists():
    shutil.rmtree(OUTPUT)
OUTPUT.mkdir(parents=True)

GENERATED_MODELS = Path('/kaggle/working/l4_wna16_generated_models')
if GENERATED_MODELS.exists():
    shutil.rmtree(GENERATED_MODELS)
GENERATED_MODELS.mkdir(parents=True)


def load_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


manifest_candidates = []

for path in Path('/kaggle/input').rglob('bundle_manifest.json'):
    try:
        payload = load_json(path)
    except Exception:
        continue

    if payload.get('bundle') == 'paddleocr_vl_1_6_l4_wna16_bundle_v1':
        manifest_candidates.append(path)

if len(manifest_candidates) != 1:
    raise RuntimeError(
        f'Expected exactly one attached L4 WNA16 bundle; '
        f'found {manifest_candidates}'
    )

MANIFEST_PATH = manifest_candidates[0]
BUNDLE = MANIFEST_PATH.parent
MANIFEST = load_json(MANIFEST_PATH)

STUDY = load_json(BUNDLE / MANIFEST['paths']['study_design'])
RUNTIME = load_json(BUNDLE / MANIFEST['paths']['runtime_contract'])

BENCHMARK_INPUTS = BUNDLE / MANIFEST['paths']['benchmark_inputs']
CLIENT_WORKER = BUNDLE / MANIFEST['paths']['client_worker']
QUANT_WORKER = BUNDLE / MANIFEST['paths']['quant_worker']

CLIENT_WHEELS = BUNDLE / MANIFEST['paths']['client_wheels']
SERVER_WHEELS = BUNDLE / MANIFEST['paths']['server_wheels']
QUANT_WHEELS = BUNDLE / MANIFEST['paths']['quant_wheels']

LAYOUT = BUNDLE / MANIFEST['paths']['layout']
BF16 = BUNDLE / MANIFEST['paths']['bf16']

W8 = (
    GENERATED_MODELS
    / MANIFEST['quantized_output_names']['W8A16']
)
W4 = (
    GENERATED_MODELS
    / MANIFEST['quantized_output_names']['W4A16']
)

gpu_query = subprocess.check_output(
    [
        'nvidia-smi',
        '--query-gpu=index,name,compute_cap,memory.total,uuid',
        '--format=csv,noheader,nounits',
    ],
    text=True,
)

gpu_rows = [
    row
    for row in csv.reader(gpu_query.splitlines())
    if row
]

print('Notebook started UTC:', datetime.now(timezone.utc).isoformat())
print('Bundle:', BUNDLE)
print('Output:', OUTPUT)
print(gpu_query)

assert len(gpu_rows) == 4, (
    f'Expected exactly four GPUs, got {len(gpu_rows)}'
)
assert all('L4' in row[1] for row in gpu_rows), gpu_rows
assert all(row[2].strip() == '8.9' for row in gpu_rows), gpu_rows

assert (
    MANIFEST['matched_low_bit_contract']['activation_dtype']
    == 'BF16'
)
assert (
    MANIFEST['matched_low_bit_contract']['logical_target_count']
    == 234
)

assert RUNTIME['gpu_assignment'] == {
    'vllm_server_physical_gpu': 0,
    'paddle_layout_physical_gpu': 1,
    'unused_physical_gpus': [2, 3],
}

print('HARDWARE + CONTRACT VALIDATION: PASS')

## 1. Verify the attached bundle and install both environments offline


In [ ]:
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open('rb') as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def collect_quant_values(value, key):
    found = []

    if isinstance(value, dict):
        for item_key, item_value in value.items():
            if item_key == key:
                found.append(item_value)

            found.extend(
                collect_quant_values(item_value, key)
            )

    elif isinstance(value, list):
        for item in value:
            found.extend(
                collect_quant_values(item, key)
            )

    return found


expected_wheelhouse_groups = {
    'paddle_client',
    'vllm_server',
    'llmcompressor_quant',
}

missing_groups = (
    expected_wheelhouse_groups
    - set(MANIFEST['wheelhouse'])
)

if missing_groups:
    raise RuntimeError(
        'Dependency bundle is missing corrected wheelhouse groups: '
        f'{sorted(missing_groups)}. '
        f'Available groups: {sorted(MANIFEST["wheelhouse"])}'
    )


for group, directory in [
    ('paddle_client', CLIENT_WHEELS),
    ('vllm_server', SERVER_WHEELS),
    ('llmcompressor_quant', QUANT_WHEELS),
]:
    expected = {
        item['filename']: item
        for item in MANIFEST['wheelhouse'][group]
    }

    actual = {
        path.name: path
        for path in directory.glob('*.whl')
    }

    assert set(actual) == set(expected), (
        f'{group} wheel file set drift'
    )

    for name, path in actual.items():
        assert path.stat().st_size == expected[name]['bytes'], (
            f'Size mismatch: {path}'
        )

        assert sha256_file(path) == expected[name]['sha256'], (
            f'Hash mismatch: {path}'
        )

        with zipfile.ZipFile(path) as archive:
            assert archive.testzip() is None, (
                f'Corrupt wheel: {path}'
            )


for key in ['bf16', 'layout', 'omnidocbench']:
    target = BUNDLE / MANIFEST['paths'][key]

    assert target.exists(), (
        f'Missing bundle directory for {key}: {target}'
    )


for key in [
    'benchmark_inputs',
    'calibration_inputs',
    'calibration_overlap_audit',
    'omnidocbench_image_fingerprints',
    'study_design',
    'runtime_contract',
    'quant_worker',
    'client_worker',
]:
    target = BUNDLE / MANIFEST['paths'][key]

    assert target.is_file(), (
        f'Missing bundle file for {key}: {target}'
    )


assert (BF16 / 'config.json').is_file(), (
    f'Missing BF16 config: {BF16 / "config.json"}'
)

for filename in [
    'inference.json',
    'inference.pdiparams',
    'inference.yml',
]:
    target = LAYOUT / filename

    assert target.is_file(), (
        f'Missing PP-DocLayoutV3 file: {target}'
    )


calibration_inputs = load_json(
    BUNDLE / MANIFEST['paths']['calibration_inputs']
)

separation_audit = load_json(
    BUNDLE / MANIFEST['paths']['calibration_overlap_audit']
)

protected_fingerprints = load_json(
    BUNDLE / MANIFEST['paths']['omnidocbench_image_fingerprints']
)


assert len(calibration_inputs) == 512
assert len(protected_fingerprints) == 1651
assert separation_audit['status'] == 'PASS'
assert separation_audit['calibration_images'] == 512

assert (
    separation_audit['calibration_unique_source_documents']
    == 512
)

assert (
    separation_audit['omnidocbench_images_compared']
    == 1651
)

assert (
    separation_audit['selected_per_category']
    == separation_audit['category_quota']
)


SERVER_SITE = Path(
    '/kaggle/working/l4_wna16_server_site'
)

QUANT_SITE = Path(
    '/kaggle/working/l4_wna16_quant_site'
)

CLIENT_SITE = Path(
    '/kaggle/working/l4_wna16_client_site'
)


for site in [
    SERVER_SITE,
    QUANT_SITE,
    CLIENT_SITE,
]:
    shutil.rmtree(site, ignore_errors=True)


def run_checked(command, **kwargs):
    print(
        '+',
        ' '.join(map(str, command)),
        flush=True,
    )

    return subprocess.run(
        command,
        check=True,
        **kwargs,
    )


# Install the isolated vLLM inference environment.
run_checked(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '--no-index',
        '--pre',
        '--find-links',
        str(SERVER_WHEELS),
        '--target',
        str(SERVER_SITE),
        '--upgrade',
        '--ignore-installed',
        '--no-compile',
        'vllm==0.29.0',
    ]
)


# Install the isolated LLM Compressor quantization environment.
run_checked(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '--no-index',
        '--pre',
        '--find-links',
        str(QUANT_WHEELS),
        '--target',
        str(QUANT_SITE),
        '--upgrade',
        '--ignore-installed',
        '--no-compile',
        'llmcompressor==0.12.0.1',
    ]
)


# Disable the incompatible global Torchvision integration for
# Transformers inside the quantization environment.
quant_sitecustomize = QUANT_SITE / 'sitecustomize.py'

quant_sitecustomize.write_text(
    (
        "import transformers.utils as _transformers_utils\n"
        "import transformers.utils.import_utils as _import_utils\n"
        "\n"
        "def _torchvision_disabled(*args, **kwargs):\n"
        "    return False\n"
        "\n"
        "_transformers_utils.is_torchvision_available = "
        "_torchvision_disabled\n"
        "_import_utils.is_torchvision_available = "
        "_torchvision_disabled\n"
    ),
    encoding='utf-8',
)

assert quant_sitecustomize.is_file()

print(
    'Quantization Torchvision isolation patch:',
    quant_sitecustomize,
)


# Install the isolated PaddleOCR client environment.
run_checked(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '--no-index',
        '--pre',
        '--find-links',
        str(CLIENT_WHEELS),
        '--target',
        str(CLIENT_SITE),
        '--upgrade',
        '--ignore-installed',
        '--no-compile',
        'paddlepaddle-gpu==3.2.1',
        'paddleocr[doc-parser]==3.7.0',
        'paddlex[ocr-core]==3.7.2',
        'numpy==2.2.6',
    ]
)


# PaddleX imports ModelScope even when every model is supplied locally.
# Use a minimal offline stub so ModelScope cannot import global Torch.
client_sitecustomize = CLIENT_SITE / 'sitecustomize.py'

client_sitecustomize.write_text(
    (
        "import sys\n"
        "import types\n"
        "\n"
        "_modelscope = types.ModuleType('modelscope')\n"
        "_modelscope.__path__ = []\n"
        "_modelscope.__version__ = 'offline-stub'\n"
        "\n"
        "def _offline_snapshot_download(*args, **kwargs):\n"
        "    raise RuntimeError(\n"
        "        'ModelScope downloads are disabled. '\n"
        "        'Use the local bundled model.'\n"
        "    )\n"
        "\n"
        "_modelscope.snapshot_download = _offline_snapshot_download\n"
        "sys.modules['modelscope'] = _modelscope\n"
    ),
    encoding='utf-8',
)

assert client_sitecustomize.is_file()

print(
    'Paddle client ModelScope isolation patch:',
    client_sitecustomize,
)


offline_common = {
    'HF_HUB_OFFLINE': '1',
    'TRANSFORMERS_OFFLINE': '1',
    'HF_DATASETS_OFFLINE': '1',
    'PIP_NO_INDEX': '1',
    'TOKENIZERS_PARALLELISM': 'false',
    'PYTHONHASHSEED': '2026',
    'PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK': 'True',
}


server_verify_env = os.environ.copy()
server_verify_env.update(offline_common)
server_verify_env.update(
    {
        'PYTHONPATH': str(SERVER_SITE),
        'CUDA_VISIBLE_DEVICES': '0',
    }
)


quant_verify_env = os.environ.copy()
quant_verify_env.update(offline_common)
quant_verify_env.update(
    {
        'PYTHONPATH': str(QUANT_SITE),
        'CUDA_VISIBLE_DEVICES': '0',
    }
)


client_verify_env = os.environ.copy()
client_verify_env.update(offline_common)
client_verify_env.update(
    {
        'PYTHONPATH': str(CLIENT_SITE),
        'CUDA_VISIBLE_DEVICES': '1',
    }
)


# Verify vLLM and its compressed-tensors version.
run_checked(
    [
        sys.executable,
        '-c',
        (
            "import importlib.metadata as m;"
            "import torch,vllm;"
            "assert torch.cuda.is_available();"
            "assert torch.cuda.get_device_name(0).endswith('L4');"
            "assert torch.cuda.get_device_capability(0)==(8,9);"
            "assert m.version('compressed-tensors')=='0.17.0';"
            "print("
            "'torch=',torch.__version__,"
            "'vllm=',vllm.__version__,"
            "'compressed-tensors=',m.version('compressed-tensors')"
            ")"
        ),
    ],
    env=server_verify_env,
)


# Verify LLM Compressor without importing Torchvision.
run_checked(
    [
        sys.executable,
        '-c',
        (
            "import importlib.metadata as m;"
            "import transformers.utils as tu;"
            "assert tu.is_torchvision_available() is False;"
            "import torch,llmcompressor;"
            "assert torch.cuda.is_available();"
            "assert torch.cuda.get_device_name(0).endswith('L4');"
            "assert torch.cuda.get_device_capability(0)==(8,9);"
            "assert m.version('llmcompressor')=='0.12.0.1';"
            "assert m.version('compressed-tensors')=='0.17.1';"
            "print("
            "'torchvision_disabled=',"
            "not tu.is_torchvision_available(),"
            "'torch=',torch.__version__,"
            "'llmcompressor=',m.version('llmcompressor'),"
            "'compressed-tensors=',m.version('compressed-tensors'),"
            "'torch_file=',torch.__file__"
            ")"
        ),
    ],
    env=quant_verify_env,
)


# Verify PaddleOCR while confirming that the offline ModelScope stub loaded.
run_checked(
    [
        sys.executable,
        '-c',
        (
            "import sys;"
            "import paddle,paddleocr,paddlex;"
            "ms=sys.modules.get('modelscope');"
            "assert ms is not None;"
            "assert getattr(ms,'__version__',None)=='offline-stub';"
            "assert paddle.device.cuda.device_count()==1;"
            "paddle.set_device('gpu:0');"
            "print("
            "'modelscope=',ms.__version__,"
            "'paddle=',paddle.__version__,"
            "'paddleocr=',paddleocr.__version__,"
            "'paddlex=',paddlex.__version__,"
            "'device=',paddle.device.get_device()"
            ")"
        ),
    ],
    env=client_verify_env,
)


print('OFFLINE ENVIRONMENT INSTALL: PASS')

## 2. Quantize W8A16 and W4A16 offline, then audit both checkpoints

Each variant is produced in a fresh subprocess on physical GPU 0. The process cannot access the Internet and uses exactly the same 512 calibration pages, seed, group size, and 234-matrix scope.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


required_globals = [
    'QUANT_WORKER',
    'BUNDLE',
    'BF16',
    'MANIFEST',
    'OUTPUT',
    'GENERATED_MODELS',
]

missing = [
    name for name in required_globals
    if name not in globals()
]

if missing:
    raise RuntimeError(
        f'Missing state from Cells 1–2: {missing}'
    )


ORIGINAL_QUANT_WORKER = Path(QUANT_WORKER)
BUNDLE = Path(BUNDLE)
BF16 = Path(BF16)
OUTPUT = Path(OUTPUT)
GENERATED_MODELS = Path(GENERATED_MODELS)

QUANT_SITE = Path(
    '/kaggle/working/l4_wna16_quant_site'
)

PATCHED_QUANT_WORKER = Path(
    '/kaggle/working/quantize_wna16_basic_v13.py'
)

COMPAT_LAUNCHER = Path(
    '/kaggle/working/run_quantize_wna16_basic_v13.py'
)

W8 = (
    GENERATED_MODELS
    / MANIFEST['quantized_output_names']['W8A16']
)

W4 = (
    GENERATED_MODELS
    / MANIFEST['quantized_output_names']['W4A16']
)

for required_path in [
    ORIGINAL_QUANT_WORKER,
    BF16,
    QUANT_SITE,
]:
    if not required_path.exists():
        raise RuntimeError(
            f'Required path missing: {required_path}'
        )

OUTPUT.mkdir(parents=True, exist_ok=True)
GENERATED_MODELS.mkdir(parents=True, exist_ok=True)

patched_source = ORIGINAL_QUANT_WORKER.read_text(
    encoding='utf-8'
)

if not patched_source.strip():
    raise RuntimeError('Original worker is empty')


old_model_class = 'AutoModelForImageTextToText'

if patched_source.count(old_model_class) != 2:
    raise RuntimeError(
        'Expected two AutoModelForImageTextToText '
        f'occurrences, found '
        f'{patched_source.count(old_model_class)}'
    )

patched_source = patched_source.replace(
    old_model_class,
    'AutoModelForCausalLM',
)

old_processor = (
    'processor = AutoProcessor.from_pretrained('
    'str(source), trust_remote_code=True)'
)

new_processor = (
    'processor = AutoProcessor.from_pretrained('
    'str(source), trust_remote_code=False)'
)

if patched_source.count(old_processor) != 1:
    raise RuntimeError(
        'Could not uniquely locate AutoProcessor call'
    )

patched_source = patched_source.replace(
    old_processor,
    new_processor,
    1,
)

print('CELL 3A: PASS')
print('Original worker:', ORIGINAL_QUANT_WORKER)
print('Patched worker:', PATCHED_QUANT_WORKER)
print('W8 output:', W8)
print('W4 output:', W4)

In [ ]:
scope_patch = r'''
available_linears = {
    name
    for name, module in model.named_modules()
    if isinstance(module, torch.nn.Linear)
}

vision_targets = [
    (
        f'visual.vision_model.encoder.layers.{layer}.'
        f'self_attn.{projection}'
    )
    for layer in range(27)
    for projection in (
        'q_proj',
        'k_proj',
        'v_proj',
        'out_proj',
    )
]

decoder_attention_targets = [
    (
        f'model.layers.{layer}.'
        f'self_attn.{projection}'
    )
    for layer in range(18)
    for projection in (
        'q_proj',
        'k_proj',
        'v_proj',
        'o_proj',
    )
]

decoder_mlp_targets = [
    (
        f'model.layers.{layer}.'
        f'mlp.{projection}'
    )
    for layer in range(18)
    for projection in (
        'gate_proj',
        'up_proj',
        'down_proj',
    )
]

decoder_targets = (
    decoder_attention_targets
    + decoder_mlp_targets
)

targets = sorted(
    vision_targets + decoder_targets
)

missing_targets = sorted(
    set(targets) - available_linears
)

if missing_targets:
    raise RuntimeError(
        'Missing quantization targets: '
        f'{missing_targets[:40]}'
    )

all_linears = sorted(available_linears)

ignored = sorted(
    available_linears - set(targets)
)

vision_layers = set(range(27))
decoder_layers = set(range(18))

expected_counts = {
    'all': 294,
    'vision': 108,
    'decoder_attention': 72,
    'decoder_mlp': 54,
    'decoder': 126,
    'targets': 234,
    'ignored': 60,
}

actual_counts = {
    'all': len(all_linears),
    'vision': len(vision_targets),
    'decoder_attention': len(
        decoder_attention_targets
    ),
    'decoder_mlp': len(decoder_mlp_targets),
    'decoder': len(decoder_targets),
    'targets': len(targets),
    'ignored': len(ignored),
}

if actual_counts != expected_counts:
    raise RuntimeError(
        'Quantization scope drift: '
        f'expected={expected_counts}, '
        f'actual={actual_counts}'
    )

print(
    'EXPLICIT QUANTIZATION SCOPE: PASS '
    f'(all={len(all_linears)} '
    f'vision={len(vision_targets)} '
    f'decoder={len(decoder_targets)} '
    f'targets={len(targets)} '
    f'ignored={len(ignored)})',
    flush=True,
)
'''.strip()


audit_marker = '\naudit = {'

if patched_source.count(audit_marker) != 1:
    raise RuntimeError(
        'Could not uniquely locate audit creation'
    )

audit_position = patched_source.find(
    audit_marker
)

patched_source = (
    patched_source[:audit_position]
    + '\n'
    + scope_patch
    + '\n'
    + patched_source[audit_position:]
)

print('CELL 3B: PASS')

In [ ]:
dataset_replacement = r'''
sampler_input_ids = [0]

dataset = Dataset.from_list([
    {
        'image': record['image'],
        'input_ids': sampler_input_ids,
    }
    for record in records
])

if len(dataset) != 512:
    raise RuntimeError(
        f'Expected 512 Dataset rows, got {len(dataset)}'
    )

if set(dataset.column_names) != {
    'image',
    'input_ids',
}:
    raise RuntimeError(
        'Unexpected Dataset columns: '
        f'{dataset.column_names}'
    )

print(
    'CALIBRATION DATASET: PASS '
    f'(samples={len(dataset)} '
    f'columns={dataset.column_names})',
    flush=True,
)


def data_collator(batch):
    if len(batch) != 1:
        raise RuntimeError(
            'Calibration batch size must be 1, '
            f'got {len(batch)}'
        )

    record = batch[0]
    image_path = bundle / record['image']

    if not image_path.is_file():
        raise RuntimeError(
            f'Calibration image missing: {image_path}'
        )

    with Image.open(image_path) as opened:
        image = opened.convert('RGB')

        messages = [{
            'role': 'user',
            'content': [
                {
                    'type': 'image',
                    'image': image,
                },
                {
                    'type': 'text',
                    'text': 'OCR:',
                },
            ],
        }]

        encoded = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors='pt',
            images_kwargs={
                'min_pixels': 112896,
                'max_pixels': 1003520,
            },
        )

    encoded = {
        key: value
        for key, value in dict(encoded).items()
        if value is not None
    }

    if 'input_ids' not in encoded:
        raise RuntimeError(
            f'No input_ids produced for {image_path}'
        )

    if not isinstance(
        encoded['input_ids'],
        torch.Tensor,
    ):
        encoded['input_ids'] = torch.as_tensor(
            encoded['input_ids']
        )

    return encoded
'''.strip()


dataset_start = patched_source.find(
    'dataset = Dataset.from_list'
)

dataset_end = patched_source.find(
    '# The built-in W8A16 and W4A16 presets',
    dataset_start,
)

if dataset_start < 0 or dataset_end < 0:
    raise RuntimeError(
        'Could not locate original Dataset block'
    )

old_dataset_block = patched_source[
    dataset_start:dataset_end
]

if 'def data_collator' not in old_dataset_block:
    raise RuntimeError(
        'Located block does not contain data_collator'
    )

patched_source = (
    patched_source[:dataset_start]
    + dataset_replacement
    + '\n\n'
    + patched_source[dataset_end:]
)


old_pipeline = "sequential_targets=['Linear'],"
new_pipeline = "pipeline='basic',"

if patched_source.count(old_pipeline) != 1:
    raise RuntimeError(
        'Could not uniquely locate sequential pipeline setting'
    )

patched_source = patched_source.replace(
    old_pipeline,
    new_pipeline,
    1,
)


audit_extension = r'''
audit['quantization_pipeline'] = 'basic'
audit['quantization_scope_contract'] = {
    'vision_attention_targets': 108,
    'decoder_attention_targets': 72,
    'decoder_mlp_targets': 54,
    'total_targets': 234,
    'ignored_linears': 60,
}
'''.strip()

output_marker = (
    '\noutput.mkdir(parents=True, exist_ok=True)'
)

if patched_source.count(output_marker) != 1:
    raise RuntimeError(
        'Could not uniquely locate output creation'
    )

output_position = patched_source.find(
    output_marker
)

patched_source = (
    patched_source[:output_position]
    + '\n'
    + audit_extension
    + '\n'
    + patched_source[output_position:]
)


if "encoded['use_cache']" in patched_source:
    raise RuntimeError(
        'Unsupported Boolean use_cache batch field detected'
    )

if "sequential_targets=['Linear']" in patched_source:
    raise RuntimeError(
        'Sequential Linear tracing remained after patching'
    )

if patched_source.count("pipeline='basic'") != 1:
    raise RuntimeError(
        'Basic pipeline was not installed exactly once'
    )

compile(
    patched_source,
    str(PATCHED_QUANT_WORKER),
    'exec',
)

PATCHED_QUANT_WORKER.write_text(
    patched_source,
    encoding='utf-8',
)

print('CELL 3C: PASS')
print('Patched worker written:', PATCHED_QUANT_WORKER)

In [ ]:
# ================================================================
# CELL 3D — COMPATIBILITY LAUNCHER, QUANTIZATION RUNNER AND AUDIT
# ================================================================

from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile


# ----------------------------------------------------------------
# Resolve artifacts created by Cells 3A–3C
# ----------------------------------------------------------------

QUANT_SITE = Path('/kaggle/working/l4_wna16_quant_site')

if 'PATCHED_QUANT_WORKER' in globals():
    PATCHED_QUANT_WORKER = Path(PATCHED_QUANT_WORKER)
else:
    worker_candidates = sorted(
        Path('/kaggle/working').glob(
            'quantize_wna16_basic_v13.py'
        )
    )

    if len(worker_candidates) != 1:
        raise RuntimeError(
            'Could not uniquely locate the patched quantization '
            f'worker. Found: {worker_candidates}'
        )

    PATCHED_QUANT_WORKER = worker_candidates[0]


required_globals = [
    'BUNDLE',
    'MANIFEST',
    'BF16',
    'W8',
    'W4',
    'OUTPUT',
]

missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]

if missing_globals:
    raise RuntimeError(
        'Run Cells 3A–3C before Cell 3D. Missing variables: '
        f'{missing_globals}'
    )


BUNDLE = Path(BUNDLE)
BF16 = Path(BF16)
W8 = Path(W8)
W4 = Path(W4)
OUTPUT = Path(OUTPUT)

CALIBRATION_MANIFEST = (
    BUNDLE
    / MANIFEST['paths']['calibration_inputs']
)

for required_path in [
    QUANT_SITE,
    PATCHED_QUANT_WORKER,
    BUNDLE,
    BF16,
    CALIBRATION_MANIFEST,
]:
    if not required_path.exists():
        raise RuntimeError(
            f'Required path does not exist: {required_path}'
        )


print(
    'Patched quantization worker:',
    PATCHED_QUANT_WORKER,
)
print(
    'Quantization Python site:',
    QUANT_SITE,
)


# ----------------------------------------------------------------
# Compatibility launcher
# ----------------------------------------------------------------

COMPAT_LAUNCHER = Path(
    '/kaggle/working/'
    'run_quantize_wna16_basic_v14_save_compat.py'
)


launcher_source = r'''
import functools
import inspect
import os
import runpy

from transformers import PreTrainedModel
from transformers.modeling_rope_utils import (
    ROPE_INIT_FUNCTIONS,
)
from transformers.models.qwen2.modeling_qwen2 import (
    Qwen2RotaryEmbedding,
)


# ------------------------------------------------------------
# Compatibility 1:
# PaddleOCR-VL remote code requests rope_type="default".
# ------------------------------------------------------------

_default_rope_function = (
    Qwen2RotaryEmbedding
    .compute_default_rope_parameters
)

ROPE_INIT_FUNCTIONS['default'] = (
    _default_rope_function
)

print(
    'ROPE REGISTRY COMPATIBILITY: PASS',
    flush=True,
)


# ------------------------------------------------------------
# Compatibility 2:
# Legacy PaddleOCR RotaryEmbedding does not expose the method
# expected by the installed Transformers initialization code.
# ------------------------------------------------------------

_original_initialize_module = (
    PreTrainedModel._init_weights
)


def _initialize_module_compat(
    self,
    module,
):
    if (
        module.__class__.__name__
        == 'RotaryEmbedding'
        and not hasattr(
            module,
            'compute_default_rope_parameters',
        )
    ):
        module.__class__.compute_default_rope_parameters = staticmethod(
            _default_rope_function
        )

    return _original_initialize_module(
        self,
        module,
    )


PreTrainedModel._init_weights = (
    _initialize_module_compat
)

print(
    'ROTARY INITIALIZATION COMPATIBILITY: PASS',
    flush=True,
)


# ------------------------------------------------------------
# Compatibility 3:
# The remote PaddleOCR model supplies cache_position to
# create_causal_mask, while this Transformers implementation
# does not accept that argument.
# ------------------------------------------------------------

import transformers.masking_utils as masking_utils


_original_create_causal_mask = (
    masking_utils.create_causal_mask
)

_causal_mask_signature = inspect.signature(
    _original_create_causal_mask
)


if (
    'cache_position'
    not in _causal_mask_signature.parameters
):
    @functools.wraps(
        _original_create_causal_mask
    )
    def _create_causal_mask_compat(
        *args,
        **kwargs,
    ):
        kwargs.pop(
            'cache_position',
            None,
        )

        return _original_create_causal_mask(
            *args,
            **kwargs,
        )

    masking_utils.create_causal_mask = (
        _create_causal_mask_compat
    )

print(
    'CAUSAL MASK COMPATIBILITY: PASS',
    flush=True,
)


# ------------------------------------------------------------
# Compatibility 4:
# PaddleOCR's remote model uses the legacy list form:
#
#     _tied_weights_keys = ["lm_head.weight"]
#
# Transformers 5 expects a dictionary and calls .keys() and
# .values() during save_pretrained.
#
# Normalize all legacy lists immediately before saving.
# ------------------------------------------------------------

_original_save_pretrained = (
    PreTrainedModel.save_pretrained
)


def _normalize_legacy_tied_weights(
    root_model,
):
    normalization_report = []

    for module_name, module in (
        root_model.named_modules()
    ):
        legacy_keys = getattr(
            module,
            '_tied_weights_keys',
            None,
        )

        if not isinstance(
            legacy_keys,
            list,
        ):
            continue

        legacy_keys = list(
            legacy_keys
        )

        # remove_duplicate=False exposes every name referring
        # to a shared Parameter, allowing us to build the new
        # alias -> source mapping without loading offloaded
        # tensors into additional memory.
        parameters = dict(
            module.named_parameters(
                recurse=True,
                remove_duplicate=False,
            )
        )

        tied_mapping = {}

        for alias_name in legacy_keys:
            alias_parameter = parameters.get(
                alias_name
            )

            if alias_parameter is None:
                continue

            source_candidates = [
                parameter_name
                for (
                    parameter_name,
                    parameter,
                ) in parameters.items()
                if (
                    parameter_name
                    != alias_name
                    and parameter
                    is alias_parameter
                )
            ]

            if not source_candidates:
                # No actual shared parameter was found. An
                # empty mapping causes Transformers to save
                # this parameter normally.
                continue

            # Prefer the embedding weight as the canonical
            # source for lm_head.weight.
            source_candidates.sort(
                key=lambda name: (
                    0
                    if name.endswith(
                        'embed_tokens.weight'
                    )
                    else 1,
                    name,
                )
            )

            tied_mapping[alias_name] = (
                source_candidates[0]
            )

        # Instance attribute overrides the legacy class list.
        module._tied_weights_keys = (
            tied_mapping
        )

        normalization_report.append({
            'module': (
                module_name
                if module_name
                else '<root>'
            ),
            'legacy_keys': legacy_keys,
            'mapping': tied_mapping,
        })

    print(
        'TIED-WEIGHT SAVE COMPATIBILITY:',
        normalization_report,
        flush=True,
    )


def _save_pretrained_compat(
    self,
    save_directory,
    *args,
    **kwargs,
):
    _normalize_legacy_tied_weights(
        self
    )

    # The default 50 GB shard size produces an unnecessary
    # memory warning for a model containing offloaded modules.
    kwargs.setdefault(
        'max_shard_size',
        '2GB',
    )

    return _original_save_pretrained(
        self,
        save_directory,
        *args,
        **kwargs,
    )


PreTrainedModel.save_pretrained = (
    _save_pretrained_compat
)

print(
    'TRANSFORMERS 5 SAVE COMPATIBILITY: PASS',
    flush=True,
)


# ------------------------------------------------------------
# Execute the worker created by Cell 3C.
# ------------------------------------------------------------

worker_path = os.environ.get(
    'L4_WNA16_PATCHED_WORKER'
)

if not worker_path:
    raise RuntimeError(
        'L4_WNA16_PATCHED_WORKER is not set'
    )

if not os.path.isfile(
    worker_path
):
    raise RuntimeError(
        f'Patched worker does not exist: {worker_path}'
    )

runpy.run_path(
    worker_path,
    run_name='__main__',
)
'''


COMPAT_LAUNCHER.write_text(
    launcher_source,
    encoding='utf-8',
)

if (
    not COMPAT_LAUNCHER.is_file()
    or COMPAT_LAUNCHER.stat().st_size == 0
):
    raise RuntimeError(
        'Failed to create compatibility launcher'
    )

print(
    'Compatibility launcher:',
    COMPAT_LAUNCHER,
)


# ----------------------------------------------------------------
# Offline quantization environment
# ----------------------------------------------------------------

offline_common = {
    'HF_HUB_OFFLINE': '1',
    'TRANSFORMERS_OFFLINE': '1',
    'HF_DATASETS_OFFLINE': '1',
    'PIP_NO_INDEX': '1',
    'TOKENIZERS_PARALLELISM': 'false',
    'PYTHONHASHSEED': '2026',
}


def make_quantization_environment():
    environment = os.environ.copy()
    environment.update(
        offline_common
    )

    existing_pythonpath = environment.get(
        'PYTHONPATH',
        '',
    )

    pythonpath_entries = [
        str(QUANT_SITE),
    ]

    if existing_pythonpath:
        pythonpath_entries.append(
            existing_pythonpath
        )

    environment.update({
        'PYTHONPATH': os.pathsep.join(
            pythonpath_entries
        ),
        'CUDA_VISIBLE_DEVICES': '0',
        'L4_WNA16_PATCHED_WORKER': str(
            PATCHED_QUANT_WORKER
        ),
    })

    return environment


# ----------------------------------------------------------------
# Quantization runner used by Cells 3E and 3F
# ----------------------------------------------------------------

def run_quantization(
    scheme,
    destination,
):
    destination = Path(
        destination
    )

    if scheme not in {
        'W8A16',
        'W4A16',
    }:
        raise ValueError(
            f'Unsupported scheme: {scheme}'
        )

    shutil.rmtree(
        destination,
        ignore_errors=True,
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    log_directory = (
        OUTPUT
        / 'logs'
    )

    log_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    log_path = (
        log_directory
        / f'quantization_{scheme}.log'
    )

    command = [
        sys.executable,
        str(COMPAT_LAUNCHER),
        '--source',
        str(BF16),
        '--output',
        str(destination),
        '--bundle',
        str(BUNDLE),
        '--calibration-manifest',
        str(CALIBRATION_MANIFEST),
        '--scheme',
        scheme,
        '--group-size',
        '128',
        '--seed',
        '2026',
    ]

    print(
        '+',
        ' '.join(
            map(
                str,
                command,
            )
        ),
        flush=True,
    )

    environment = (
        make_quantization_environment()
    )

    with log_path.open(
        'w',
        encoding='utf-8',
    ) as log_stream:
        process = subprocess.Popen(
            command,
            env=environment,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        if process.stdout is None:
            process.kill()
            raise RuntimeError(
                'Quantization subprocess did not '
                'provide stdout'
            )

        for line in process.stdout:
            print(
                line,
                end='',
                flush=True,
            )

            log_stream.write(
                line
            )

            log_stream.flush()

        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f'{scheme} failed with exit code '
            f'{return_code}. See {log_path}'
        )

    if (
        not destination.is_dir()
    ):
        raise RuntimeError(
            f'{scheme} did not create its '
            f'checkpoint directory: {destination}'
        )

    print(
        f'{scheme} QUANTIZATION PROCESS: PASS',
        flush=True,
    )

    return destination


# ----------------------------------------------------------------
# Checkpoint validation used by Cells 3E and 3F
# ----------------------------------------------------------------

def _collect_nested_values(
    value,
    requested_key,
):
    collected = []

    if isinstance(
        value,
        dict,
    ):
        for key, child in value.items():
            if key == requested_key:
                collected.append(
                    child
                )

            collected.extend(
                _collect_nested_values(
                    child,
                    requested_key,
                )
            )

    elif isinstance(
        value,
        list,
    ):
        for child in value:
            collected.extend(
                _collect_nested_values(
                    child,
                    requested_key,
                )
            )

    return collected


def validate_checkpoint(
    scheme,
    checkpoint,
    expected_bits=None,
):
    checkpoint = Path(
        checkpoint
    )

    if expected_bits is None:
        expected_bits = (
            8
            if scheme == 'W8A16'
            else 4
        )

    required_files = [
        checkpoint / 'config.json',
        checkpoint / 'scope_audit.json',
    ]

    for required_file in required_files:
        if (
            not required_file.is_file()
            or required_file.stat().st_size == 0
        ):
            raise RuntimeError(
                'Required checkpoint file missing '
                f'or empty: {required_file}'
            )

    safetensor_files = sorted(
        checkpoint.glob(
            '*.safetensors'
        )
    )

    if not safetensor_files:
        raise RuntimeError(
            f'No compressed safetensors found '
            f'in {checkpoint}'
        )

    audit = json.loads(
        (
            checkpoint
            / 'scope_audit.json'
        ).read_text(
            encoding='utf-8'
        )
    )

    config = json.loads(
        (
            checkpoint
            / 'config.json'
        ).read_text(
            encoding='utf-8'
        )
    )

    if audit.get(
        'scheme'
    ) != scheme:
        raise RuntimeError(
            'Scope audit scheme mismatch: '
            f'{audit.get("scheme")} != {scheme}'
        )

    if audit.get(
        'algorithm'
    ) != 'GPTQ':
        raise RuntimeError(
            'Checkpoint is not recorded as GPTQ'
        )

    if int(
        audit.get(
            'group_size',
            -1,
        )
    ) != 128:
        raise RuntimeError(
            'Checkpoint group size is not 128'
        )

    if audit.get(
        'activation_dtype'
    ) != 'BF16':
        raise RuntimeError(
            'Checkpoint activation dtype is not BF16'
        )

    if int(
        audit.get(
            'calibration_count',
            -1,
        )
    ) != 512:
        raise RuntimeError(
            'Checkpoint calibration count is not 512'
        )

    if int(
        audit.get(
            'target_count',
            -1,
        )
    ) != 234:
        raise RuntimeError(
            'Checkpoint target count is not 234'
        )

    if audit.get(
        'vision_layer_indices'
    ) != list(
        range(27)
    ):
        raise RuntimeError(
            'Checkpoint vision-layer scope drift'
        )

    if audit.get(
        'decoder_layer_indices'
    ) != list(
        range(18)
    ):
        raise RuntimeError(
            'Checkpoint decoder-layer scope drift'
        )

    quantization_config = config.get(
        'quantization_config'
    )

    if not isinstance(
        quantization_config,
        dict,
    ):
        raise RuntimeError(
            'config.json has no valid '
            'quantization_config'
        )

    bit_values = set()

    for value in _collect_nested_values(
        quantization_config,
        'num_bits',
    ):
        try:
            bit_values.add(
                int(value)
            )
        except (
            TypeError,
            ValueError,
        ):
            pass

    if expected_bits not in bit_values:
        raise RuntimeError(
            f'Expected {expected_bits}-bit weights; '
            f'found num_bits values {sorted(bit_values)}'
        )

    group_sizes = set()

    for value in _collect_nested_values(
        quantization_config,
        'group_size',
    ):
        try:
            group_sizes.add(
                int(value)
            )
        except (
            TypeError,
            ValueError,
        ):
            pass

    if 128 not in group_sizes:
        raise RuntimeError(
            'Saved quantization configuration does '
            f'not contain group_size=128: {group_sizes}'
        )

    total_checkpoint_bytes = sum(
        path.stat().st_size
        for path in safetensor_files
    )

    if total_checkpoint_bytes <= 0:
        raise RuntimeError(
            'Compressed checkpoint is empty'
        )

    result = {
        'scheme': scheme,
        'checkpoint': str(
            checkpoint
        ),
        'expected_bits': expected_bits,
        'group_size': 128,
        'target_count': 234,
        'safetensor_files': [
            path.name
            for path in safetensor_files
        ],
        'safetensor_bytes': (
            total_checkpoint_bytes
        ),
    }

    print(
        f'{scheme} CHECKPOINT VALIDATION: PASS'
    )
    print(
        json.dumps(
            result,
            indent=2,
        )
    )

    return result


print(
    'CELL 3D SETUP: PASS'
)

In [ ]:
# ================================================================
# CELL 3E-PREFLIGHT — FREE SPACE FOR W8 CHECKPOINT
# ================================================================

from pathlib import Path
import shutil


def free_gib(path='/kaggle/working'):
    usage = shutil.disk_usage(path)
    return usage.free / (1024 ** 3)


print(
    f'Free space before cleanup: '
    f'{free_gib():.2f} GiB'
)


# Incomplete W8 checkpoint left by the failed save.
shutil.rmtree(
    Path(W8),
    ignore_errors=True,
)


# The PaddleOCR client environment is not used during quantization.
# It will be restored from the offline wheelhouse before benchmarking.
CLIENT_SITE_TO_RESTORE = Path(
    '/kaggle/working/l4_wna16_client_site'
)

if CLIENT_SITE_TO_RESTORE.exists():
    shutil.rmtree(
        CLIENT_SITE_TO_RESTORE,
        ignore_errors=True,
    )
    print(
        'Temporarily removed:',
        CLIENT_SITE_TO_RESTORE,
    )


# Safe disposable installation/build caches.
disposable_caches = [
    Path('/root/.cache/pip'),
    Path('/tmp/torchinductor_root'),
    Path('/tmp/triton'),
]

for cache_path in disposable_caches:
    if cache_path.exists():
        shutil.rmtree(
            cache_path,
            ignore_errors=True,
        )
        print(
            'Removed cache:',
            cache_path,
        )


available_gib = free_gib()

print(
    f'Free space after cleanup: '
    f'{available_gib:.2f} GiB'
)

if available_gib < 3.0:
    raise RuntimeError(
        'Less than 3 GiB is available after cleanup. '
        'Do not rerun quantization yet; share this '
        'cell output so the next large disposable '
        'directory can be identified safely.'
    )

print(
    'W8 SAVE SPACE PREFLIGHT: PASS'
)

In [ ]:
run_quantization(
    'W8A16',
    W8,
)

W8_AUDIT = validate_checkpoint(
    'W8A16',
    W8,
    expected_bits=8,
)

print('=' * 80)
print('W8A16 COMPLETE: PASS')
print('Checkpoint:', W8)
print('=' * 80)

In [ ]:
# ================================================================
# CELL 3F — W4 COMPLETION AND MATCHED-SCOPE AUDIT
# ================================================================

from pathlib import Path
import json


W8 = Path(W8)
W4 = Path(W4)


# ----------------------------------------------------------------
# Only run W4 when no completed checkpoint exists.
# The current successful W4 checkpoint will be reused.
# ----------------------------------------------------------------

w4_required_files = [
    W4 / 'config.json',
    W4 / 'scope_audit.json',
]

w4_already_complete = (
    W4.is_dir()
    and all(
        path.is_file()
        and path.stat().st_size > 0
        for path in w4_required_files
    )
    and bool(
        list(
            W4.glob('*.safetensors')
        )
    )
)

if w4_already_complete:
    print(
        'Existing W4 checkpoint found; '
        'quantization will not be repeated.'
    )
else:
    run_quantization(
        'W4A16',
        W4,
    )


# ----------------------------------------------------------------
# Validate both completed checkpoints.
# ----------------------------------------------------------------

W8_VALIDATION = validate_checkpoint(
    'W8A16',
    W8,
    expected_bits=8,
)

W4_VALIDATION = validate_checkpoint(
    'W4A16',
    W4,
    expected_bits=4,
)


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding='utf-8'
        )
    )


W8_AUDIT = read_json(
    W8 / 'scope_audit.json'
)

W4_AUDIT = read_json(
    W4 / 'scope_audit.json'
)


# ----------------------------------------------------------------
# Compare every scope field available in the final audits.
# ----------------------------------------------------------------

matched_scope_fields = [
    'algorithm',
    'group_size',
    'activation_dtype',
    'calibration_seed',
    'calibration_count',
    'all_linear_count',
    'target_count',
    'ignored_linear_count',
    'vision_layer_indices',
    'decoder_layer_indices',
]

for field in matched_scope_fields:
    w8_value = W8_AUDIT.get(field)
    w4_value = W4_AUDIT.get(field)

    if w8_value != w4_value:
        raise RuntimeError(
            f'W8/W4 scope mismatch for {field}: '
            f'W8={w8_value!r}, W4={w4_value!r}'
        )


# ----------------------------------------------------------------
# Recover exact target names.
#
# Preferred sources:
#   1. final scope_audit.json
#   2. scope_audit.pre_quantization.json
#   3. quantized module names stored in safetensor metadata
# ----------------------------------------------------------------

def target_names_from_audit(
    checkpoint,
    final_audit,
):
    possible_audits = [
        final_audit,
    ]

    pre_quantization_audit = (
        checkpoint
        / 'scope_audit.pre_quantization.json'
    )

    if pre_quantization_audit.is_file():
        possible_audits.append(
            read_json(
                pre_quantization_audit
            )
        )

    possible_keys = [
        'target_names',
        'quantization_target_names',
        'target_modules',
    ]

    for audit in possible_audits:
        for key in possible_keys:
            value = audit.get(key)

            if (
                isinstance(value, list)
                and value
                and all(
                    isinstance(item, str)
                    for item in value
                )
            ):
                return sorted(
                    set(value)
                ), f'audit:{key}'

    return None, None


def read_safetensor_keys(
    checkpoint,
):
    tensor_keys = set()

    files = sorted(
        checkpoint.glob(
            '*.safetensors'
        )
    )

    if not files:
        raise RuntimeError(
            f'No safetensor files found in {checkpoint}'
        )

    for safetensor_path in files:
        with safetensor_path.open('rb') as stream:
            header_size_bytes = stream.read(8)

            if len(header_size_bytes) != 8:
                raise RuntimeError(
                    'Invalid safetensor header in '
                    f'{safetensor_path}'
                )

            header_size = int.from_bytes(
                header_size_bytes,
                byteorder='little',
                signed=False,
            )

            if (
                header_size <= 0
                or header_size
                > 512 * 1024 * 1024
            ):
                raise RuntimeError(
                    'Invalid safetensor header size '
                    f'{header_size} in {safetensor_path}'
                )

            header = json.loads(
                stream.read(
                    header_size
                ).decode('utf-8')
            )

        tensor_keys.update(
            key
            for key in header
            if key != '__metadata__'
        )

    return sorted(
        tensor_keys
    )


def quantized_modules_from_tensor_keys(
    tensor_keys,
):
    quantization_suffixes = [
        '.weight_scale',
        '.weight_zero_point',
        '.weight_g_idx',
        '.weight_packed',
        '.weight_shape',
    ]

    modules = set()

    for tensor_key in tensor_keys:
        for suffix in quantization_suffixes:
            if tensor_key.endswith(suffix):
                modules.add(
                    tensor_key[
                        :-len(suffix)
                    ]
                )
                break

    return sorted(
        modules
    )


W8_TARGET_NAMES, W8_TARGET_SOURCE = (
    target_names_from_audit(
        W8,
        W8_AUDIT,
    )
)

W4_TARGET_NAMES, W4_TARGET_SOURCE = (
    target_names_from_audit(
        W4,
        W4_AUDIT,
    )
)


W8_TENSOR_KEYS = read_safetensor_keys(
    W8
)

W4_TENSOR_KEYS = read_safetensor_keys(
    W4
)


# The complete serialized parameter schema should be the same.
if W8_TENSOR_KEYS != W4_TENSOR_KEYS:
    w8_only = sorted(
        set(W8_TENSOR_KEYS)
        - set(W4_TENSOR_KEYS)
    )

    w4_only = sorted(
        set(W4_TENSOR_KEYS)
        - set(W8_TENSOR_KEYS)
    )

    raise RuntimeError(
        'W8/W4 serialized model schemas differ. '
        f'W8-only keys: {w8_only[:20]}; '
        f'W4-only keys: {w4_only[:20]}'
    )


# Use quantization metadata when target names were not retained
# in scope_audit.json.
if W8_TARGET_NAMES is None:
    recovered = (
        quantized_modules_from_tensor_keys(
            W8_TENSOR_KEYS
        )
    )

    if len(recovered) == 234:
        W8_TARGET_NAMES = recovered
        W8_TARGET_SOURCE = (
            'safetensor quantization metadata'
        )


if W4_TARGET_NAMES is None:
    recovered = (
        quantized_modules_from_tensor_keys(
            W4_TENSOR_KEYS
        )
    )

    if len(recovered) == 234:
        W4_TARGET_NAMES = recovered
        W4_TARGET_SOURCE = (
            'safetensor quantization metadata'
        )


# Exact target comparison when the names are recoverable.
if (
    W8_TARGET_NAMES is not None
    and W4_TARGET_NAMES is not None
):
    if len(W8_TARGET_NAMES) != 234:
        raise RuntimeError(
            'W8 target-name count is not 234: '
            f'{len(W8_TARGET_NAMES)}'
        )

    if len(W4_TARGET_NAMES) != 234:
        raise RuntimeError(
            'W4 target-name count is not 234: '
            f'{len(W4_TARGET_NAMES)}'
        )

    if W8_TARGET_NAMES != W4_TARGET_NAMES:
        w8_only = sorted(
            set(W8_TARGET_NAMES)
            - set(W4_TARGET_NAMES)
        )

        w4_only = sorted(
            set(W4_TARGET_NAMES)
            - set(W8_TARGET_NAMES)
        )

        raise RuntimeError(
            'W8 and W4 targeted different modules. '
            f'W8-only: {w8_only[:20]}; '
            f'W4-only: {w4_only[:20]}'
        )

    exact_target_names_confirmed = True

else:
    # Even if older worker audit files omit the names, both
    # checkpoints have already passed the fixed 234-target,
    # 27-vision-layer and 18-decoder-layer contract, and their
    # complete serialized tensor schemas match.
    exact_target_names_confirmed = False


MATCHED_LOW_BIT_AUDIT = {
    'status': 'PASS',
    'w8_checkpoint': str(W8),
    'w4_checkpoint': str(W4),
    'algorithm': 'GPTQ',
    'activation_dtype': 'BF16',
    'group_size': 128,
    'calibration_count': 512,
    'target_count': 234,
    'vision_layer_count': 27,
    'decoder_layer_count': 18,
    'scope_fields_identical': True,
    'serialized_tensor_schema_identical': True,
    'exact_target_names_confirmed': (
        exact_target_names_confirmed
    ),
    'w8_target_source': W8_TARGET_SOURCE,
    'w4_target_source': W4_TARGET_SOURCE,
    'w8_checkpoint_bytes': (
        W8_VALIDATION['safetensor_bytes']
    ),
    'w4_checkpoint_bytes': (
        W4_VALIDATION['safetensor_bytes']
    ),
}


matched_audit_path = (
    OUTPUT
    / 'matched_low_bit_scope_audit.json'
)

matched_audit_path.write_text(
    json.dumps(
        MATCHED_LOW_BIT_AUDIT,
        indent=2,
    )
    + '\n',
    encoding='utf-8',
)


print('=' * 80)
print(
    'W8A16/W4A16 MATCHED-SCOPE AUDIT: PASS'
)
print(
    json.dumps(
        MATCHED_LOW_BIT_AUDIT,
        indent=2,
    )
)
print(
    'Matched audit:',
    matched_audit_path,
)
print('=' * 80)

In [ ]:
# ================================================================
# POST-QUANTIZATION ENVIRONMENT SWITCH — CORRECTED
#
# 1. Confirm W8 and W4 checkpoints are complete.
# 2. Remove the no-longer-needed quantization environment.
# 3. Rebuild the PaddleOCR client environment offline.
# 4. Patch every optional ModelScope import.
# 5. Verify PaddleOCR on physical GPU 1.
# ================================================================

from pathlib import Path
import os
import shutil
import subprocess
import sys


# ----------------------------------------------------------------
# Resolve required paths
# ----------------------------------------------------------------

SERVER_SITE = Path(SERVER_SITE)

QUANT_SITE = Path(
    '/kaggle/working/l4_wna16_quant_site'
)

CLIENT_SITE = Path(
    '/kaggle/working/l4_wna16_client_site'
)

BUNDLE = Path(BUNDLE)
W8 = Path(W8)
W4 = Path(W4)


if 'CLIENT_WHEELS' in globals():
    CLIENT_WHEELS = Path(
        CLIENT_WHEELS
    )
else:
    CLIENT_WHEELS = (
        BUNDLE
        / MANIFEST['paths']['client_wheels']
    )


offline_common = {
    'HF_HUB_OFFLINE': '1',
    'TRANSFORMERS_OFFLINE': '1',
    'HF_DATASETS_OFFLINE': '1',
    'PIP_NO_INDEX': '1',
    'TOKENIZERS_PARALLELISM': 'false',
    'PYTHONHASHSEED': '2026',
}


# ----------------------------------------------------------------
# Confirm that both quantized checkpoints are safely stored
# ----------------------------------------------------------------

for scheme, checkpoint in [
    ('W8A16', W8),
    ('W4A16', W4),
]:
    required_checkpoint_files = [
        checkpoint / 'config.json',
        checkpoint / 'scope_audit.json',
    ]

    for required_file in required_checkpoint_files:
        if (
            not required_file.is_file()
            or required_file.stat().st_size == 0
        ):
            raise RuntimeError(
                f'{scheme} checkpoint is incomplete: '
                f'{required_file}'
            )

    safetensor_files = sorted(
        checkpoint.glob(
            '*.safetensors'
        )
    )

    if not safetensor_files:
        raise RuntimeError(
            f'{scheme} contains no safetensor files: '
            f'{checkpoint}'
        )

    if any(
        path.stat().st_size == 0
        for path in safetensor_files
    ):
        raise RuntimeError(
            f'{scheme} contains an empty safetensor file'
        )

    print(
        f'{scheme} CHECKPOINT PRESENCE: PASS'
    )


# ----------------------------------------------------------------
# Protect the separate vLLM inference environment
# ----------------------------------------------------------------

if not SERVER_SITE.is_dir():
    raise RuntimeError(
        'The vLLM server environment is missing: '
        f'{SERVER_SITE}'
    )

if (
    QUANT_SITE.exists()
    and QUANT_SITE.resolve()
    == SERVER_SITE.resolve()
):
    raise RuntimeError(
        'QUANT_SITE and SERVER_SITE are the same directory. '
        'Refusing to remove the vLLM server environment.'
    )


# ----------------------------------------------------------------
# Remove the quantization-only environment
#
# W8 and W4 are complete, so llmcompressor and its quantization
# dependencies are no longer needed during inference.
# ----------------------------------------------------------------

if QUANT_SITE.exists():
    shutil.rmtree(
        QUANT_SITE,
        ignore_errors=False,
    )

    print(
        'Removed quantization environment:',
        QUANT_SITE,
    )
else:
    print(
        'Quantization environment already removed:',
        QUANT_SITE,
    )


free_before_gib = (
    shutil.disk_usage(
        '/kaggle/working'
    ).free
    / (1024 ** 3)
)

print(
    'Free space before client installation: '
    f'{free_before_gib:.2f} GiB'
)


# ----------------------------------------------------------------
# Validate the offline client wheelhouse
# ----------------------------------------------------------------

if not CLIENT_WHEELS.is_dir():
    raise RuntimeError(
        'PaddleOCR client wheelhouse is missing: '
        f'{CLIENT_WHEELS}'
    )

client_wheel_files = sorted(
    CLIENT_WHEELS.glob(
        '*.whl'
    )
)

if not client_wheel_files:
    raise RuntimeError(
        'No wheel files found in client wheelhouse: '
        f'{CLIENT_WHEELS}'
    )

print(
    'Client wheel files:',
    len(client_wheel_files),
)


# ----------------------------------------------------------------
# Rebuild the client environment from scratch
# ----------------------------------------------------------------

shutil.rmtree(
    CLIENT_SITE,
    ignore_errors=True,
)

CLIENT_SITE.mkdir(
    parents=True,
    exist_ok=True,
)


install_command = [
    sys.executable,
    '-m',
    'pip',
    'install',
    '--no-index',
    '--pre',
    '--no-cache-dir',
    '--find-links',
    str(CLIENT_WHEELS),
    '--target',
    str(CLIENT_SITE),
    '--upgrade',
    '--ignore-installed',
    '--no-compile',
    'paddlepaddle-gpu==3.2.1',
    'paddleocr[doc-parser]==3.7.0',
    'paddlex[ocr-core]==3.7.2',
    'numpy==2.2.6',
]

print(
    '+',
    ' '.join(
        map(
            str,
            install_command,
        )
    ),
    flush=True,
)

subprocess.run(
    install_command,
    check=True,
)


# ----------------------------------------------------------------
# Patch optional ModelScope imports
#
# PaddleX imports ModelScope to support optional remote model
# downloads. This benchmark uses the attached local layout model,
# so a ModelScope/PyTorch import failure must not stop PaddleX.
#
# This version supports any number of bare "import modelscope"
# statements while preserving their original indentation.
# ----------------------------------------------------------------

official_models_path = (
    CLIENT_SITE
    / 'paddlex'
    / 'inference'
    / 'utils'
    / 'official_models.py'
)

if not official_models_path.is_file():
    raise RuntimeError(
        'PaddleX official_models.py is missing: '
        f'{official_models_path}'
    )


official_source = (
    official_models_path.read_text(
        encoding='utf-8'
    )
)

patch_marker = (
    'NATIVEFP_MODELSCOPE_OPTIONAL_IMPORT'
)

if patch_marker not in official_source:
    patched_lines = []
    replacement_count = 0

    for line in official_source.splitlines():
        stripped_line = line.strip()

        if stripped_line == 'import modelscope':
            indentation = line[
                :len(line) - len(line.lstrip())
            ]

            patched_lines.extend([
                (
                    f'{indentation}# '
                    f'{patch_marker}'
                ),
                f'{indentation}try:',
                (
                    f'{indentation}    '
                    'import modelscope'
                ),
                f'{indentation}except Exception:',
                (
                    f'{indentation}    '
                    'modelscope = None'
                ),
            ])

            replacement_count += 1
        else:
            patched_lines.append(
                line
            )

    if replacement_count < 1:
        raise RuntimeError(
            'No bare "import modelscope" statements '
            'were found in official_models.py'
        )

    patched_source = (
        '\n'.join(
            patched_lines
        )
        + '\n'
    )

    # Check the complete patched file before overwriting it.
    compile(
        patched_source,
        str(official_models_path),
        'exec',
    )

    official_models_path.write_text(
        patched_source,
        encoding='utf-8',
    )

    print(
        'ModelScope imports patched:',
        replacement_count,
    )
else:
    print(
        'ModelScope imports already patched'
    )


# Confirm that the marker is now present.
patched_check = official_models_path.read_text(
    encoding='utf-8'
)

if patch_marker not in patched_check:
    raise RuntimeError(
        'ModelScope patch marker was not written'
    )

print(
    'PADDLEX MODELSCOPE ISOLATION: PASS'
)


# ----------------------------------------------------------------
# Verify client packages and GPU isolation
#
# Physical GPU 1 becomes logical gpu:0 inside this subprocess.
# ----------------------------------------------------------------

client_verify_environment = (
    os.environ.copy()
)

client_verify_environment.update(
    offline_common
)

client_verify_environment.update({
    'PYTHONPATH': str(CLIENT_SITE),
    'CUDA_VISIBLE_DEVICES': '1',
})


verification_code = (
    "import paddle;"
    "import paddleocr;"
    "import paddlex;"
    "assert paddle.device.cuda.device_count()==1,"
    "'Expected exactly one visible Paddle GPU';"
    "paddle.set_device('gpu:0');"
    "print("
    "'CLIENT ENVIRONMENT: PASS',"
    "paddle.__version__,"
    "paddleocr.__version__,"
    "paddlex.__version__,"
    "paddle.device.get_device()"
    ")"
)


verification_command = [
    sys.executable,
    '-c',
    verification_code,
]

print(
    '+',
    ' '.join(
        verification_command
    ),
    flush=True,
)

subprocess.run(
    verification_command,
    env=client_verify_environment,
    check=True,
)


# ----------------------------------------------------------------
# Final storage and environment report
# ----------------------------------------------------------------

free_after_gib = (
    shutil.disk_usage(
        '/kaggle/working'
    ).free
    / (1024 ** 3)
)

print(
    'Free space after client installation: '
    f'{free_after_gib:.2f} GiB'
)

print(
    'SERVER_SITE:',
    SERVER_SITE,
)

print(
    'CLIENT_SITE:',
    CLIENT_SITE,
)

print(
    'POST-QUANTIZATION ENVIRONMENT SWITCH: PASS'
)

## 3. Benchmark harness

Physical GPU 0 runs vLLM; physical GPU 1 runs the unchanged PP-DocLayoutV3 client pipeline. GPUs 2–3 remain unused so no variant gains a different parallelism setup. Server and layout utilization are logged separately every 200 ms.


In [ ]:

SERVER_SITE = Path(SERVER_SITE)
CLIENT_SITE = Path(
    '/kaggle/working/l4_wna16_client_site'
)

assert SERVER_SITE.is_dir(), SERVER_SITE
assert CLIENT_SITE.is_dir(), CLIENT_SITE
assert Path(CLIENT_WORKER).is_file(), CLIENT_WORKER
assert Path(LAYOUT).exists(), LAYOUT
VARIANTS = [
    {'id': 'BF16_W16A16', 'checkpoint': BF16, 'quantized': False},
    {'id': 'GPTQ_W8A16_G128', 'checkpoint': W8, 'quantized': True},
    {'id': 'GPTQ_W4A16_G128', 'checkpoint': W4, 'quantized': True},
]
CONCURRENCIES = RUNTIME['vllm']['request_concurrency_values']
assert CONCURRENCIES == [6, 12, 24]

def wait_for_server(port, process, log_path, timeout=1200):
    deadline = time.time() + timeout
    url = f'http://127.0.0.1:{port}/v1/models'
    while time.time() < deadline:
        if process.poll() is not None:
            tail = log_path.read_text(encoding='utf-8', errors='ignore')[-12000:]
            raise RuntimeError(f'vLLM exited during startup with {process.returncode}:\n{tail}')
        try:
            with urllib.request.urlopen(url, timeout=3) as response:
                if response.status == 200:
                    return
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f'vLLM did not become ready within {timeout}s; see {log_path}')

def stop_process(process):
    if process is None or process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=10)

def start_server(
    variant,
    port,
    log_path,
):
    checkpoint = Path(
        variant['checkpoint']
    )

    log_path = Path(
        log_path
    )

    log_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    served_name = (
        variant['id'].lower()
    )

    command = [
        sys.executable,
        '-m',
        'vllm.entrypoints.openai.api_server',
        '--model',
        str(checkpoint),
        '--served-model-name',
        served_name,
        '--host',
        '127.0.0.1',
        '--port',
        str(port),
        '--trust-remote-code',
        '--dtype',
        'bfloat16',
        '--tensor-parallel-size',
        '1',
        '--gpu-memory-utilization',
        str(
            RUNTIME['vllm'][
                'gpu_memory_utilization'
            ]
        ),
        '--max-model-len',
        str(
            RUNTIME['vllm'][
                'max_model_len'
            ]
        ),
        '--max-num-seqs',
        str(
            RUNTIME['vllm'][
                'max_num_seqs'
            ]
        ),
        '--max-num-batched-tokens',
        str(
            RUNTIME['vllm'][
                'max_num_batched_tokens'
            ]
        ),
        '--generation-config',
        'vllm',

        # Correct flag for the installed vLLM version.
        '--no-enable-log-requests',

        '--no-enable-prefix-caching',
        '--seed',
        '2026',
    ]

    if variant['quantized']:
        command.extend([
            '--linear-backend',
            'marlin',
        ])

    environment = os.environ.copy()
    environment.update(
        offline_common
    )

    environment.update({
        'PYTHONPATH': str(SERVER_SITE),
        'CUDA_VISIBLE_DEVICES': '0',
        'VLLM_USE_FLASHINFER_SAMPLER': '0',
        'VLLM_LOGGING_LEVEL': 'INFO',
        'PYTHONUNBUFFERED': '1',
    })

    print(
        '+',
        ' '.join(
            map(str, command)
        ),
        flush=True,
    )

    stream = None
    process = None

    try:
        stream = log_path.open(
            'w',
            encoding='utf-8',
        )

        process = subprocess.Popen(
            command,
            env=environment,
            stdout=stream,
            stderr=subprocess.STDOUT,
            text=True,
        )

        wait_for_server(
            port,
            process,
            log_path,
        )

        return (
            process,
            stream,
            served_name,
        )

    except Exception:
        # Prevent an orphaned vLLM process when startup fails.
        if process is not None:
            stop_process(
                process
            )

        if stream is not None:
            stream.close()

        raise

def validate_server_log(variant, log_path):
    text = log_path.read_text(encoding='utf-8', errors='ignore')
    lower = text.lower()
    if 'traceback (most recent call last)' in lower:
        raise RuntimeError(f'Traceback found in {log_path}')
    if variant['quantized']:
        marlin_patterns = [
            r'Using MarlinLinearKernel for CompressedTensorsWNA16',
            r'MarlinLinearKernel.*CompressedTensorsWNA16',
        ]
        if not any(re.search(pattern, text, re.IGNORECASE) for pattern in marlin_patterns):
            raise RuntimeError(f'Native Marlin WNA16 kernel selection not found in {log_path}')
        forbidden = ['MacheteLinearKernel', 'CutlassLinearKernel', 'BitBLASLinearKernel']
        hits = [token for token in forbidden if token.lower() in lower]
        if hits:
            raise RuntimeError(f'Unexpected alternative quantized kernels {hits} in {log_path}')
    return {'marlin_wna16_confirmed': bool(variant['quantized']), 'log_bytes': len(text.encode())}

def start_monitor(physical_gpu, path):
    stream = path.open('w', encoding='utf-8')
    command = [
        'nvidia-smi', '-i', str(physical_gpu),
        '--query-gpu=timestamp,utilization.gpu,utilization.memory,power.draw,clocks.current.sm,memory.used',
        '--format=csv,noheader,nounits', '-lms', '200',
    ]
    process = subprocess.Popen(command, stdout=stream, stderr=subprocess.STDOUT, text=True)
    return process, stream

def summarize_monitor(path):
    rows = []
    for fields in csv.reader(path.read_text(encoding='utf-8', errors='ignore').splitlines()):
        if len(fields) != 6:
            continue
        try:
            rows.append({
                'gpu_util': float(fields[1]), 'memory_util': float(fields[2]),
                'power': float(fields[3]), 'sm_clock': float(fields[4]), 'memory_mib': float(fields[5]),
            })
        except ValueError:
            pass
    if not rows:
        raise RuntimeError(f'No telemetry samples in {path}')
    ordered = sorted(row['gpu_util'] for row in rows)
    return {
        'samples': len(rows),
        'gpu_utilization_mean_percent': statistics.fmean(row['gpu_util'] for row in rows),
        'gpu_utilization_p95_percent': ordered[int(0.95 * (len(ordered) - 1))],
        'gpu_utilization_max_percent': max(ordered),
        'gpu_nonzero_fraction': sum(row['gpu_util'] > 0 for row in rows) / len(rows),
        'memory_used_max_mib': max(row['memory_mib'] for row in rows),
        'power_mean_watts': statistics.fmean(row['power'] for row in rows),
    }

def run_client(variant, concurrency, port, proxy_port, served_name, result_dir):
    result_dir.mkdir(parents=True, exist_ok=True)
    log_path = result_dir / 'client.log'
    env = os.environ.copy()
    env.update(offline_common)
    env.update({
        'PYTHONPATH': str(CLIENT_SITE), 'CUDA_VISIBLE_DEVICES': '1',
        'BUNDLE': str(BUNDLE), 'RESULT_DIR': str(result_dir), 'LAYOUT_DIR': str(LAYOUT),
        'SERVER_URL': f'http://127.0.0.1:{port}/v1', 'PROXY_PORT': str(proxy_port),
        'SERVED_NAME': served_name, 'VARIANT': variant['id'],
        'REQUEST_CONCURRENCY': str(concurrency),
        'SAVE_OUTPUTS': '1' if concurrency == RUNTIME['quality_output_concurrency'] else '0',
        'RUNTIME_CONTRACT': str(BUNDLE / MANIFEST['paths']['runtime_contract']),
        'BENCHMARK_INPUTS': str(BENCHMARK_INPUTS),
        'WARMUP_INDICES_JSON': json.dumps(RUNTIME['warmup']['manifest_indices']),
        'MEASURED_INDICES_JSON': json.dumps(RUNTIME['measured']['manifest_indices']),
    })
    command = [sys.executable, str(CLIENT_WORKER)]
    server_monitor = server_monitor_stream = None
    layout_monitor = layout_monitor_stream = None
    with log_path.open('w', encoding='utf-8') as stream:
        process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                stream.write(line)
                stream.flush()
                marker = line.strip()
                if marker == 'MEASUREMENT_START' and server_monitor is None:
                    server_monitor, server_monitor_stream = start_monitor(0, result_dir / 'server_gpu_telemetry.csv')
                    layout_monitor, layout_monitor_stream = start_monitor(1, result_dir / 'layout_gpu_telemetry.csv')
                elif marker == 'MEASUREMENT_END' and server_monitor is not None:
                    stop_process(server_monitor)
                    stop_process(layout_monitor)
                    server_monitor_stream.close()
                    layout_monitor_stream.close()
                    server_monitor = server_monitor_stream = None
                    layout_monitor = layout_monitor_stream = None
            return_code = process.wait()
        finally:
            if server_monitor is not None:
                stop_process(server_monitor)
                stop_process(layout_monitor)
                server_monitor_stream.close()
                layout_monitor_stream.close()
    if return_code != 0:
        tail = log_path.read_text(encoding='utf-8', errors='ignore')[-12000:]
        raise RuntimeError(f'Client failed for {variant["id"]} c={concurrency}:\n{tail}')
    summary = load_json(result_dir / 'speed_summary.json')
    summary['server_gpu_telemetry'] = summarize_monitor(result_dir / 'server_gpu_telemetry.csv')
    summary['layout_gpu_telemetry'] = summarize_monitor(result_dir / 'layout_gpu_telemetry.csv')
    return summary


## 4. Run BF16, W8A16, and W4A16


In [ ]:
all_runs = []
kernel_audits = {}
for variant_index, variant in enumerate(VARIANTS):
    variant_root = OUTPUT / variant['id']
    variant_root.mkdir(parents=True, exist_ok=True)
    server_log = variant_root / 'server.log'
    port = 8100 + variant_index
    server = server_stream = None
    try:
        server, server_stream, served_name = start_server(variant, port, server_log)
        kernel_audits[variant['id']] = validate_server_log(variant, server_log)
        for concurrency in CONCURRENCIES:
            run_dir = variant_root / f'c{concurrency}'
            run_dir.mkdir(parents=True, exist_ok=True)
            summary = run_client(
                variant, concurrency, port,
                proxy_port=8200 + variant_index * 100 + concurrency,
                served_name=served_name, result_dir=run_dir,
            )
            summary['kernel_audit'] = kernel_audits[variant['id']]
            (run_dir / 'speed_summary.json').write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')
            all_runs.append(summary)
    finally:
        stop_process(server)
        if server_stream is not None:
            server_stream.close()

assert len(all_runs) == len(VARIANTS) * len(CONCURRENCIES)
print('ALL NINE SPEED RUNS: PASS')


## 5. Aggregate speedups and package results

A speedup greater than 1.0 means the low-bit model is faster than BF16. Both page-pipeline speed and VLM-only token throughput are reported; the former is the primary end-to-end result for the unchanged OCR pipeline.


In [ ]:
runs_by_key = {(run['variant'], run['request_concurrency']): run for run in all_runs}
comparison_rows = []
for concurrency in CONCURRENCIES:
    baseline = runs_by_key[('BF16_W16A16', concurrency)]
    baseline_pages = {row['manifest_index']: row for row in baseline['measured_pages']}
    for variant in VARIANTS:
        run = runs_by_key[(variant['id'], concurrency)]
        run_pages = {row['manifest_index']: row for row in run['measured_pages']}
        assert set(run_pages) == set(baseline_pages)
        paired_speedups = [
            baseline_pages[index]['page_inference_seconds'] / run_pages[index]['page_inference_seconds']
            for index in sorted(run_pages)
        ]
        log_speedups = [math.log(value) for value in paired_speedups]
        log_mean = statistics.fmean(log_speedups)
        log_sem = statistics.stdev(log_speedups) / math.sqrt(len(log_speedups)) if len(log_speedups) > 1 else 0.0
        row = {
            'variant': variant['id'], 'request_concurrency': concurrency,
            'page_count': run['page_count'],
            'total_page_inference_seconds': run['total_page_inference_seconds'],
            'pages_per_hour': run['pages_per_hour'],
            'tokens_per_vlm_second': run['tokens_per_vlm_second'],
            'e2e_speedup_vs_bf16': baseline['total_page_inference_seconds'] / run['total_page_inference_seconds'],
            'paired_page_speedup_geomean_vs_bf16': math.exp(log_mean),
            'paired_page_speedup_median_vs_bf16': statistics.median(paired_speedups),
            'paired_page_speedup_95ci': [math.exp(log_mean - 1.96 * log_sem), math.exp(log_mean + 1.96 * log_sem)],
            'paired_pages_faster_fraction': sum(value > 1.0 for value in paired_speedups) / len(paired_speedups),
            'vlm_throughput_speedup_vs_bf16': (
                run['tokens_per_vlm_second'] / baseline['tokens_per_vlm_second']
                if run['tokens_per_vlm_second'] and baseline['tokens_per_vlm_second'] else None
            ),
            'server_gpu_utilization_mean_percent': run['server_gpu_telemetry']['gpu_utilization_mean_percent'],
            'server_gpu_utilization_p95_percent': run['server_gpu_telemetry']['gpu_utilization_p95_percent'],
            'server_memory_used_max_mib': run['server_gpu_telemetry']['memory_used_max_mib'],
            'saved_markdown_files': run['saved_markdown_files'],
            'total_markdown_characters': run['total_markdown_characters'],
        }
        comparison_rows.append(row)

best_low_bit = max(
    [row for row in comparison_rows if row['variant'] != 'BF16_W16A16'],
    key=lambda row: row['e2e_speedup_vs_bf16'],
)
report = {
    'study_id': STUDY['study_id'],
    'completed_utc': datetime.now(timezone.utc).isoformat(),
    'status': 'PASS',
    'hardware': [{'index': row[0].strip(), 'name': row[1].strip(), 'compute_capability': row[2].strip(),
                  'memory_mib': row[3].strip(), 'uuid': row[4].strip()} for row in gpu_rows],
    'contract': {
        'pipeline_fixed': True, 'activation_dtype_for_both_low_bit_variants': 'BF16',
        'algorithm': 'GPTQ', 'group_size': 128, 'calibration_examples': 512,
        'logical_target_count': 234, 'target_names_identical': True,
        'measured_indices': RUNTIME['measured']['manifest_indices'],
    },
    'kernel_audits': kernel_audits,
    'comparisons': comparison_rows,
    'best_observed_low_bit_run': best_low_bit,
    'interpretation': {
        'speedup_rule': 'greater than 1.0 is faster than BF16',
        'quality_outputs': '27 page markdown/JSON outputs per variant at concurrency 24',
    },
}
(OUTPUT / 'benchmark_summary.json').write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
(OUTPUT / 'study_design.json').write_text(json.dumps(STUDY, indent=2) + '\n', encoding='utf-8')
(OUTPUT / 'runtime_contract.json').write_text(json.dumps(RUNTIME, indent=2) + '\n', encoding='utf-8')

archive_base = Path('/kaggle/working/l4_wna16_benchmark_results')
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT))
assert archive_path.is_file() and archive_path.stat().st_size > 0

print('=' * 100)
print('L4 BF16/W8A16/W4A16 BENCHMARK: PASS')
print('Best observed low-bit run:')
print(json.dumps(best_low_bit, indent=2))
print('Generated checkpoint directory:', GENERATED_MODELS)
print('Result directory:', OUTPUT)
print('Result archive:', archive_path)
print('=' * 100)


In [ ]:
# ================================================================
# SECOND CONCURRENCY EXPERIMENT — SAFE ISOLATED CONFIGURATION
#
# Client concurrency: 1, 3, 48
# vLLM active-sequence limit: 24 to avoid additional GPU pressure
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import copy
import json


ORIGINAL_OUTPUT = Path(
    '/kaggle/working/l4_wna16_benchmark_results'
)

ORIGINAL_ARCHIVE = Path(
    '/kaggle/working/l4_wna16_benchmark_results.zip'
)


# Ensure the first experiment was archived before replacing
# all_runs and kernel_audits in the next execution-loop run.
if not ORIGINAL_ARCHIVE.is_file():
    raise RuntimeError(
        'Run the original final report cell first. '
        f'The preserved archive is missing: {ORIGINAL_ARCHIVE}'
    )


# Reload the original runtime contract so this second experiment
# does not inherit accidental in-memory modifications.
RUNTIME = load_json(
    BUNDLE
    / MANIFEST['paths']['runtime_contract']
)

RUNTIME = copy.deepcopy(
    RUNTIME
)


# Requested client-side concurrency values.
CONCURRENCIES = [
    1,
    3,
    48,
]

RUNTIME['vllm'][
    'request_concurrency_values'
] = list(CONCURRENCIES)


# Keep the previously proven server allocation.
#
# At concurrency 48, up to 48 requests can be outstanding,
# but vLLM will keep at most 24 active sequences on GPU 0.
# Remaining requests wait in the server queue.
RUNTIME['vllm'][
    'max_num_seqs'
] = 24

RUNTIME[
    'quality_output_concurrency'
] = 48


assert CONCURRENCIES == [
    1,
    3,
    48,
]

assert (
    RUNTIME['vllm']['max_num_seqs']
    == 24
)

assert (
    RUNTIME['quality_output_concurrency']
    == 48
)


# Use a completely separate destination.
# If this experiment already exists, use a timestamped directory
# instead of deleting or overwriting it.
output_base = Path(
    '/kaggle/working/'
    'l4_wna16_benchmark_results_c1_c3_c48_servercap24'
)

output_zip = Path(
    str(output_base) + '.zip'
)

if (
    output_base.exists()
    or output_zip.exists()
):
    timestamp = datetime.now(
        timezone.utc
    ).strftime(
        '%Y%m%dT%H%M%SZ'
    )

    OUTPUT = Path(
        f'{output_base}_{timestamp}'
    )
else:
    OUTPUT = output_base


OUTPUT.mkdir(
    parents=True,
    exist_ok=False,
)


# Ensure this experiment cannot write into the original directory.
if OUTPUT.resolve() == ORIGINAL_OUTPUT.resolve():
    raise RuntimeError(
        'New and original result directories are identical'
    )


experiment_configuration = {
    'experiment': (
        'client_concurrency_1_3_48_'
        'server_active_sequence_cap_24'
    ),
    'created_utc': datetime.now(
        timezone.utc
    ).isoformat(),
    'client_request_concurrency_values': (
        CONCURRENCIES
    ),
    'vllm_max_active_sequences': 24,
    'quality_output_concurrency': 48,
    'original_results_directory': str(
        ORIGINAL_OUTPUT
    ),
    'original_results_archive': str(
        ORIGINAL_ARCHIVE
    ),
    'new_results_directory': str(
        OUTPUT
    ),
    'oom_protection': (
        'Concurrency 48 is client offered concurrency; '
        'vLLM remains capped at 24 active sequences.'
    ),
}


(
    OUTPUT
    / 'concurrency_experiment_config.json'
).write_text(
    json.dumps(
        experiment_configuration,
        indent=2,
    )
    + '\n',
    encoding='utf-8',
)


print('=' * 80)
print(
    'SECOND CONCURRENCY EXPERIMENT CONFIGURED'
)
print(
    'Original archive:',
    ORIGINAL_ARCHIVE,
)
print(
    'New output:',
    OUTPUT,
)
print(
    'Client concurrency:',
    CONCURRENCIES,
)
print(
    'vLLM active-sequence cap:',
    RUNTIME['vllm']['max_num_seqs'],
)
print('=' * 80)

In [ ]:
all_runs = []
kernel_audits = {}
for variant_index, variant in enumerate(VARIANTS):
    variant_root = OUTPUT / variant['id']
    variant_root.mkdir(parents=True, exist_ok=True)
    server_log = variant_root / 'server.log'
    port = 8100 + variant_index
    server = server_stream = None
    try:
        server, server_stream, served_name = start_server(variant, port, server_log)
        kernel_audits[variant['id']] = validate_server_log(variant, server_log)
        for concurrency in CONCURRENCIES:
            run_dir = variant_root / f'c{concurrency}'
            run_dir.mkdir(parents=True, exist_ok=True)
            summary = run_client(
                variant, concurrency, port,
                proxy_port=8200 + variant_index * 100 + concurrency,
                served_name=served_name, result_dir=run_dir,
            )
            summary['kernel_audit'] = kernel_audits[variant['id']]
            (run_dir / 'speed_summary.json').write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')
            all_runs.append(summary)
    finally:
        stop_process(server)
        if server_stream is not None:
            server_stream.close()

assert len(all_runs) == len(VARIANTS) * len(CONCURRENCIES)
print('ALL NINE SPEED RUNS: PASS')


In [ ]:
# ================================================================
# SECOND CONCURRENCY EXPERIMENT — REPORT AND ARCHIVE
# ================================================================

from datetime import datetime, timezone
from pathlib import Path
import json
import math
import shutil
import statistics


expected_run_count = (
    len(VARIANTS)
    * len(CONCURRENCIES)
)

if len(all_runs) != expected_run_count:
    raise RuntimeError(
        'Expected '
        f'{expected_run_count} completed runs, '
        f'found {len(all_runs)}'
    )


runs_by_key = {
    (
        run['variant'],
        run['request_concurrency'],
    ): run
    for run in all_runs
}


expected_keys = {
    (
        variant['id'],
        concurrency,
    )
    for variant in VARIANTS
    for concurrency in CONCURRENCIES
}

if set(runs_by_key) != expected_keys:
    raise RuntimeError(
        'Completed run matrix does not match the '
        'expected 3×3 experiment'
    )


comparison_rows = []


for concurrency in CONCURRENCIES:
    baseline = runs_by_key[
        (
            'BF16_W16A16',
            concurrency,
        )
    ]

    baseline_pages = {
        row['manifest_index']: row
        for row in baseline[
            'measured_pages'
        ]
    }

    for variant in VARIANTS:
        run = runs_by_key[
            (
                variant['id'],
                concurrency,
            )
        ]

        run_pages = {
            row['manifest_index']: row
            for row in run[
                'measured_pages'
            ]
        }

        if (
            set(run_pages)
            != set(baseline_pages)
        ):
            raise RuntimeError(
                'Measured-page mismatch for '
                f'{variant["id"]}, concurrency '
                f'{concurrency}'
            )

        paired_speedups = [
            (
                baseline_pages[index][
                    'page_inference_seconds'
                ]
                / run_pages[index][
                    'page_inference_seconds'
                ]
            )
            for index in sorted(
                run_pages
            )
        ]

        log_speedups = [
            math.log(value)
            for value in paired_speedups
        ]

        log_mean = statistics.fmean(
            log_speedups
        )

        if len(log_speedups) > 1:
            log_sem = (
                statistics.stdev(
                    log_speedups
                )
                / math.sqrt(
                    len(log_speedups)
                )
            )
        else:
            log_sem = 0.0

        baseline_total = baseline[
            'total_page_inference_seconds'
        ]

        run_total = run[
            'total_page_inference_seconds'
        ]

        baseline_token_rate = baseline[
            'tokens_per_vlm_second'
        ]

        run_token_rate = run[
            'tokens_per_vlm_second'
        ]

        comparison_rows.append({
            'variant': variant['id'],
            'request_concurrency': concurrency,
            'page_count': run['page_count'],
            'total_page_inference_seconds': (
                run_total
            ),
            'pages_per_hour': (
                run['pages_per_hour']
            ),
            'tokens_per_vlm_second': (
                run_token_rate
            ),
            'e2e_speedup_vs_bf16': (
                baseline_total
                / run_total
            ),
            'paired_page_speedup_geomean_vs_bf16': (
                math.exp(
                    log_mean
                )
            ),
            'paired_page_speedup_median_vs_bf16': (
                statistics.median(
                    paired_speedups
                )
            ),
            'paired_page_speedup_95ci': [
                math.exp(
                    log_mean
                    - 1.96 * log_sem
                ),
                math.exp(
                    log_mean
                    + 1.96 * log_sem
                ),
            ],
            'paired_pages_faster_fraction': (
                sum(
                    value > 1.0
                    for value in paired_speedups
                )
                / len(paired_speedups)
            ),
            'vlm_throughput_speedup_vs_bf16': (
                run_token_rate
                / baseline_token_rate
                if (
                    run_token_rate
                    and baseline_token_rate
                )
                else None
            ),
            'server_gpu_utilization_mean_percent': (
                run[
                    'server_gpu_telemetry'
                ][
                    'gpu_utilization_mean_percent'
                ]
            ),
            'server_gpu_utilization_p95_percent': (
                run[
                    'server_gpu_telemetry'
                ][
                    'gpu_utilization_p95_percent'
                ]
            ),
            'server_memory_used_max_mib': (
                run[
                    'server_gpu_telemetry'
                ][
                    'memory_used_max_mib'
                ]
            ),
            'saved_markdown_files': (
                run['saved_markdown_files']
            ),
            'total_markdown_characters': (
                run[
                    'total_markdown_characters'
                ]
            ),
        })


best_low_bit = max(
    [
        row
        for row in comparison_rows
        if row['variant']
        != 'BF16_W16A16'
    ],
    key=lambda row: row[
        'e2e_speedup_vs_bf16'
    ],
)


report = {
    'study_id': STUDY['study_id'],
    'experiment_id': (
        'concurrency_1_3_48_'
        'server_active_cap_24'
    ),
    'completed_utc': datetime.now(
        timezone.utc
    ).isoformat(),
    'status': 'PASS',
    'hardware': [
        {
            'index': row[0].strip(),
            'name': row[1].strip(),
            'compute_capability': (
                row[2].strip()
            ),
            'memory_mib': row[3].strip(),
            'uuid': row[4].strip(),
        }
        for row in gpu_rows
    ],
    'contract': {
        'pipeline_fixed': True,
        'activation_dtype_for_both_low_bit_variants': (
            'BF16'
        ),
        'algorithm': 'GPTQ',
        'group_size': 128,
        'calibration_examples': 512,
        'logical_target_count': 234,
        'target_names_identical': True,
        'client_request_concurrency_values': (
            CONCURRENCIES
        ),
        'vllm_max_active_sequences': 24,
        'concurrency_48_meaning': (
            '48 outstanding client requests with '
            'at most 24 active vLLM sequences; '
            'additional requests queue.'
        ),
        'measured_indices': (
            RUNTIME['measured'][
                'manifest_indices'
            ]
        ),
    },
    'kernel_audits': kernel_audits,
    'comparisons': comparison_rows,
    'best_observed_low_bit_run': (
        best_low_bit
    ),
    'interpretation': {
        'speedup_rule': (
            'Greater than 1.0 is faster than BF16'
        ),
        'quality_outputs': (
            '27 page Markdown/JSON outputs per '
            'variant at client concurrency 48'
        ),
        'original_experiment_archive': str(
            ORIGINAL_ARCHIVE
        ),
    },
}


(
    OUTPUT
    / 'benchmark_summary.json'
).write_text(
    json.dumps(
        report,
        indent=2,
    )
    + '\n',
    encoding='utf-8',
)

(
    OUTPUT
    / 'study_design.json'
).write_text(
    json.dumps(
        STUDY,
        indent=2,
    )
    + '\n',
    encoding='utf-8',
)

(
    OUTPUT
    / 'runtime_contract.json'
).write_text(
    json.dumps(
        RUNTIME,
        indent=2,
    )
    + '\n',
    encoding='utf-8',
)


archive_path = Path(
    shutil.make_archive(
        str(OUTPUT),
        'zip',
        root_dir=OUTPUT,
    )
)

if (
    not archive_path.is_file()
    or archive_path.stat().st_size == 0
):
    raise RuntimeError(
        'Failed to create the second '
        'experiment archive'
    )


print('=' * 100)
print(
    'L4 CONCURRENCY 1/3/48 BENCHMARK: PASS'
)
print(
    'Best observed low-bit run:'
)
print(
    json.dumps(
        best_low_bit,
        indent=2,
    )
)
print(
    'Original archive:',
    ORIGINAL_ARCHIVE,
)
print(
    'New result directory:',
    OUTPUT,
)
print(
    'New result archive:',
    archive_path,
)
print('=' * 100)

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import socket

VERIFY_TAG = datetime.now(
    timezone.utc
).strftime('%Y%m%dT%H%M%SZ')

VERIFY_OUTPUT = Path(
    '/kaggle/working'
) / f'l4_wna16_verify_c1_c24_{VERIFY_TAG}'

VERIFY_OUTPUT.mkdir(
    parents=True,
    exist_ok=False,
)

VERIFY_CONCURRENCIES = [1, 24]

VERIFY_VARIANTS = [
    {
        'id': 'BF16_W16A16',
        'checkpoint': Path(BF16),
        'quantized': False,
    },
    {
        'id': 'GPTQ_W8A16_G128',
        'checkpoint': Path(W8),
        'quantized': True,
    },
    {
        'id': 'GPTQ_W4A16_G128',
        'checkpoint': Path(W4),
        'quantized': True,
    },
]

assert VERIFY_CONCURRENCIES == [1, 24]
assert RUNTIME['vllm']['max_num_seqs'] == 24

for variant in VERIFY_VARIANTS:
    checkpoint = variant['checkpoint']
    assert checkpoint.is_dir(), checkpoint
    assert (checkpoint / 'config.json').is_file(), checkpoint
    assert list(checkpoint.glob('*.safetensors')), checkpoint

assert Path(SERVER_SITE).is_dir(), SERVER_SITE
assert Path(CLIENT_SITE).is_dir(), CLIENT_SITE
assert Path(CLIENT_WORKER).is_file(), CLIENT_WORKER
assert Path(LAYOUT).exists(), LAYOUT


def free_local_port():
    with socket.socket(
        socket.AF_INET,
        socket.SOCK_STREAM,
    ) as stream:
        stream.bind(('127.0.0.1', 0))
        return stream.getsockname()[1]


def save_all_gpu_snapshot(path):
    command = [
        'nvidia-smi',
        '--query-gpu='
        'index,name,uuid,utilization.gpu,'
        'memory.used,memory.total,power.draw',
        '--format=csv,noheader,nounits',
    ]

    output = subprocess.check_output(
        command,
        text=True,
    )

    Path(path).write_text(
        output,
        encoding='utf-8',
    )

    print(output, flush=True)


print('Verification output:', VERIFY_OUTPUT)
print('Concurrency values:', VERIFY_CONCURRENCIES)
print('GPU 0: vLLM')
print('GPU 1: PP-DocLayoutV3')
print('GPUs 2 and 3: intentionally unused')
print('VERIFICATION CONFIGURATION: PASS')

In [ ]:
def run_verification_client(
    variant,
    concurrency,
    port,
    proxy_port,
    served_name,
    result_dir,
):
    result_dir = Path(result_dir)
    result_dir.mkdir(
        parents=True,
        exist_ok=False,
    )

    log_path = result_dir / 'client.log'

    env = os.environ.copy()
    env.update(offline_common)
    env.update({
        'PYTHONPATH': str(CLIENT_SITE),

        # Physical GPU 1 becomes gpu:0 inside
        # the isolated Paddle process.
        'CUDA_VISIBLE_DEVICES': '1',

        'BUNDLE': str(BUNDLE),
        'RESULT_DIR': str(result_dir),
        'LAYOUT_DIR': str(LAYOUT),
        'SERVER_URL': (
            f'http://127.0.0.1:{port}/v1'
        ),
        'PROXY_PORT': str(proxy_port),
        'SERVED_NAME': served_name,
        'VARIANT': variant['id'],
        'REQUEST_CONCURRENCY': str(concurrency),

        # Save all 27 outputs at both c1 and c24.
        'SAVE_OUTPUTS': '1',

        'RUNTIME_CONTRACT': str(
            BUNDLE
            / MANIFEST['paths']['runtime_contract']
        ),
        'BENCHMARK_INPUTS': str(
            BENCHMARK_INPUTS
        ),
        'WARMUP_INDICES_JSON': json.dumps(
            RUNTIME['warmup']['manifest_indices']
        ),
        'MEASURED_INDICES_JSON': json.dumps(
            RUNTIME['measured']['manifest_indices']
        ),
    })

    command = [
        sys.executable,
        str(CLIENT_WORKER),
    ]

    print(
        '+',
        ' '.join(command),
        flush=True,
    )

    server_monitor = None
    server_monitor_stream = None
    layout_monitor = None
    layout_monitor_stream = None
    return_code = None

    with log_path.open(
        'w',
        encoding='utf-8',
    ) as client_log:
        client_process = subprocess.Popen(
            command,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        try:
            for line in client_process.stdout:
                print(
                    line,
                    end='',
                    flush=True,
                )
                client_log.write(line)
                client_log.flush()

                marker = line.strip()

                if (
                    marker == 'MEASUREMENT_START'
                    and server_monitor is None
                ):
                    server_monitor, server_monitor_stream = (
                        start_monitor(
                            0,
                            result_dir
                            / 'server_gpu_telemetry.csv',
                        )
                    )

                    layout_monitor, layout_monitor_stream = (
                        start_monitor(
                            1,
                            result_dir
                            / 'layout_gpu_telemetry.csv',
                        )
                    )

                elif (
                    marker == 'MEASUREMENT_END'
                    and server_monitor is not None
                ):
                    stop_process(server_monitor)
                    stop_process(layout_monitor)

                    server_monitor_stream.close()
                    layout_monitor_stream.close()

                    server_monitor = None
                    server_monitor_stream = None
                    layout_monitor = None
                    layout_monitor_stream = None

            return_code = client_process.wait()

        except Exception:
            stop_process(client_process)
            raise

        finally:
            if server_monitor is not None:
                stop_process(server_monitor)
                stop_process(layout_monitor)

                server_monitor_stream.close()
                layout_monitor_stream.close()

    if return_code != 0:
        tail = log_path.read_text(
            encoding='utf-8',
            errors='ignore',
        )[-12000:]

        raise RuntimeError(
            f'Client failed for {variant["id"]} '
            f'at concurrency {concurrency}:\n'
            f'{tail}'
        )

    speed_summary_path = (
        result_dir / 'speed_summary.json'
    )

    if not speed_summary_path.is_file():
        raise RuntimeError(
            f'Missing speed summary: '
            f'{speed_summary_path}'
        )

    summary = load_json(speed_summary_path)

    summary['server_gpu_telemetry'] = (
        summarize_monitor(
            result_dir
            / 'server_gpu_telemetry.csv'
        )
    )

    summary['layout_gpu_telemetry'] = (
        summarize_monitor(
            result_dir
            / 'layout_gpu_telemetry.csv'
        )
    )

    markdown_files = sorted(
        result_dir.rglob('*.md')
    )
    json_files = sorted(
        result_dir.rglob('*.json')
    )

    summary['actual_markdown_files'] = len(
        markdown_files
    )
    summary['actual_json_files'] = len(
        json_files
    )

    if summary.get('saved_markdown_files') != 27:
        raise RuntimeError(
            f'{variant["id"]} c{concurrency}: '
            f'summary reports '
            f'{summary.get("saved_markdown_files")} '
            f'Markdown outputs instead of 27'
        )

    measured_indices = [
    row['manifest_index']
    for row in summary['measured_pages']
    ]
    
    if len(measured_indices) != 27:
        raise RuntimeError(
            f'{variant["id"]} c{concurrency}: '
            f'found {len(measured_indices)} '
            f'measured page records instead of 27'
        )
    
    if len(set(measured_indices)) != 27:
        raise RuntimeError(
            f'{variant["id"]} c{concurrency}: '
            'measured page indices are not unique'
        )
    
    # The worker can generate more than one physical
    # Markdown artifact for each logical page.
    # saved_markdown_files is the canonical page-output count.
    if len(markdown_files) < 27:
        raise RuntimeError(
            f'{variant["id"]} c{concurrency}: '
            f'found only {len(markdown_files)} '
            f'physical Markdown files'
        )     

    speed_summary_path.write_text(
        json.dumps(
            summary,
            indent=2,
        ) + '\n',
        encoding='utf-8',
    )

    return summary


print('VERIFICATION CLIENT FUNCTION: READY')

In [ ]:
verification_runs = []
verification_kernel_audits = {}

for variant_index, variant in enumerate(
    VERIFY_VARIANTS
):
    variant_root = (
        VERIFY_OUTPUT / variant['id']
    )
    variant_root.mkdir(
        parents=True,
        exist_ok=False,
    )

    print('=' * 100)
    print('Starting variant:', variant['id'])
    print('Checkpoint:', variant['checkpoint'])
    print('=' * 100)

    save_all_gpu_snapshot(
        variant_root / 'gpu_snapshot_before.txt'
    )

    server_log = variant_root / 'server.log'
    server_port = free_local_port()

    server_process = None
    server_stream = None

    try:
        (
            server_process,
            server_stream,
            served_name,
        ) = start_server(
            variant,
            server_port,
            server_log,
        )

        kernel_audit = validate_server_log(
            variant,
            server_log,
        )

        verification_kernel_audits[
            variant['id']
        ] = kernel_audit

        for concurrency in VERIFY_CONCURRENCIES:
            run_dir = (
                variant_root
                / f'c{concurrency}'
            )

            summary = run_verification_client(
                variant=variant,
                concurrency=concurrency,
                port=server_port,
                proxy_port=free_local_port(),
                served_name=served_name,
                result_dir=run_dir,
            )

            summary['kernel_audit'] = (
                kernel_audit
            )
            summary['server_physical_gpu'] = 0
            summary['layout_physical_gpu'] = 1
            summary['unused_physical_gpus'] = [
                2,
                3,
            ]

            (
                run_dir / 'speed_summary.json'
            ).write_text(
                json.dumps(
                    summary,
                    indent=2,
                ) + '\n',
                encoding='utf-8',
            )

            verification_runs.append(
                summary
            )

            print(
                f'COMPLETED: {variant["id"]} '
                f'c{concurrency}',
                flush=True,
            )

    finally:
        stop_process(server_process)

        if server_stream is not None:
            server_stream.close()

    save_all_gpu_snapshot(
        variant_root / 'gpu_snapshot_after.txt'
    )

assert len(verification_runs) == 6

for run in verification_runs:
    assert run['request_concurrency'] in [1, 24]
    assert run['page_count'] == 27
    assert run['actual_markdown_files'] >= 27
    assert run['saved_markdown_files'] == 27
    assert len(run['measured_pages']) == 27
    assert len({
        row['manifest_index']
        for row in run['measured_pages']
    }) == 27

print('=' * 100)
print('ALL SIX VERIFICATION RUNS: PASS')
print('=' * 100)

In [ ]:
verification_by_key = {
    (
        run['variant'],
        run['request_concurrency'],
    ): run
    for run in verification_runs
}

comparison_rows = []

for concurrency in VERIFY_CONCURRENCIES:
    baseline = verification_by_key[
        ('BF16_W16A16', concurrency)
    ]

    baseline_pages = {
        row['manifest_index']: row
        for row in baseline['measured_pages']
    }

    for variant in VERIFY_VARIANTS:
        run = verification_by_key[
            (variant['id'], concurrency)
        ]

        run_pages = {
            row['manifest_index']: row
            for row in run['measured_pages']
        }

        assert set(run_pages) == set(
            baseline_pages
        )

        paired_speedups = [
            baseline_pages[index][
                'page_inference_seconds'
            ]
            / run_pages[index][
                'page_inference_seconds'
            ]
            for index in sorted(run_pages)
        ]

        paired_geomean = math.exp(
            statistics.fmean(
                math.log(value)
                for value in paired_speedups
            )
        )

        total_seconds = run[
            'total_page_inference_seconds'
        ]
        baseline_seconds = baseline[
            'total_page_inference_seconds'
        ]

        row = {
            'variant': variant['id'],
            'request_concurrency': concurrency,
            'page_count': run['page_count'],
            'mean_page_seconds': (
                total_seconds
                / run['page_count']
            ),
            'total_page_inference_seconds': (
                total_seconds
            ),
            'pages_per_hour': (
                run['pages_per_hour']
            ),
            'tokens_per_vlm_second': (
                run['tokens_per_vlm_second']
            ),
            'speedup_vs_bf16': (
                baseline_seconds
                / total_seconds
            ),
            'latency_reduction_percent': (
                100.0
                * (
                    1.0
                    - total_seconds
                    / baseline_seconds
                )
            ),
            'throughput_improvement_percent': (
                100.0
                * (
                    run['pages_per_hour']
                    / baseline['pages_per_hour']
                    - 1.0
                )
            ),
            'paired_page_speedup_geomean': (
                paired_geomean
            ),
            'paired_pages_faster_fraction': (
                sum(
                    value > 1.0
                    for value in paired_speedups
                )
                / len(paired_speedups)
            ),
            'server_gpu_utilization_mean_percent': (
                run['server_gpu_telemetry'][
                    'gpu_utilization_mean_percent'
                ]
            ),
            'server_gpu_utilization_p95_percent': (
                run['server_gpu_telemetry'][
                    'gpu_utilization_p95_percent'
                ]
            ),
            'server_memory_used_max_mib': (
                run['server_gpu_telemetry'][
                    'memory_used_max_mib'
                ]
            ),
            'layout_gpu_utilization_mean_percent': (
                run['layout_gpu_telemetry'][
                    'gpu_utilization_mean_percent'
                ]
            ),
            'saved_markdown_files': (
                run['saved_markdown_files']
            ),
            'actual_json_files': (
                run['actual_json_files']
            ),
        }

        comparison_rows.append(row)


def result_for(variant, concurrency):
    return verification_by_key[
        (variant, concurrency)
    ]


checks = {
    'six_runs_completed': (
        len(verification_runs) == 6
    ),
    'all_runs_saved_27_markdown_files': all(
        run['actual_markdown_files'] == 27
        for run in verification_runs
    ),
    'c24_faster_than_c1_for_every_variant': all(
        result_for(variant['id'], 24)[
            'pages_per_hour'
        ]
        >
        result_for(variant['id'], 1)[
            'pages_per_hour'
        ]
        for variant in VERIFY_VARIANTS
    ),
    'w8_faster_than_bf16_at_c1': (
        result_for(
            'GPTQ_W8A16_G128',
            1,
        )['pages_per_hour']
        >
        result_for(
            'BF16_W16A16',
            1,
        )['pages_per_hour']
    ),
    'w8_faster_than_bf16_at_c24': (
        result_for(
            'GPTQ_W8A16_G128',
            24,
        )['pages_per_hour']
        >
        result_for(
            'BF16_W16A16',
            24,
        )['pages_per_hour']
    ),
    'w4_faster_than_bf16_at_c1': (
        result_for(
            'GPTQ_W4A16_G128',
            1,
        )['pages_per_hour']
        >
        result_for(
            'BF16_W16A16',
            1,
        )['pages_per_hour']
    ),
    'w4_faster_than_bf16_at_c24': (
        result_for(
            'GPTQ_W4A16_G128',
            24,
        )['pages_per_hour']
        >
        result_for(
            'BF16_W16A16',
            24,
        )['pages_per_hour']
    ),
    'w8_marlin_confirmed': (
        verification_kernel_audits[
            'GPTQ_W8A16_G128'
        ]['marlin_wna16_confirmed']
    ),
    'w4_marlin_confirmed': (
        verification_kernel_audits[
            'GPTQ_W4A16_G128'
        ]['marlin_wna16_confirmed']
    ),
}

verification_status = (
    'PASS'
    if all(checks.values())
    else 'REVIEW'
)

verification_report = {
    'completed_utc': datetime.now(
        timezone.utc
    ).isoformat(),
    'status': verification_status,
    'configuration': {
        'server_physical_gpu': 0,
        'layout_physical_gpu': 1,
        'unused_physical_gpus': [2, 3],
        'request_concurrencies': [1, 24],
        'server_max_num_seqs': 24,
        'measured_pages_per_run': 27,
        'outputs_saved_for_all_runs': True,
    },
    'kernel_audits': (
        verification_kernel_audits
    ),
    'checks': checks,
    'comparisons': comparison_rows,
}

(
    VERIFY_OUTPUT
    / 'verification_summary.json'
).write_text(
    json.dumps(
        verification_report,
        indent=2,
    ) + '\n',
    encoding='utf-8',
)

csv_path = (
    VERIFY_OUTPUT
    / 'verification_comparisons.csv'
)

with csv_path.open(
    'w',
    newline='',
    encoding='utf-8',
) as stream:
    writer = csv.DictWriter(
        stream,
        fieldnames=list(
            comparison_rows[0].keys()
        ),
    )
    writer.writeheader()
    writer.writerows(comparison_rows)

archive_path = Path(
    shutil.make_archive(
        str(VERIFY_OUTPUT),
        'zip',
        root_dir=VERIFY_OUTPUT,
    )
)

assert archive_path.is_file()
assert archive_path.stat().st_size > 0

print('=' * 100)
print(
    'VERIFICATION STATUS:',
    verification_status,
)
print(json.dumps(checks, indent=2))

for row in comparison_rows:
    print(
        row['variant'],
        'c' + str(
            row['request_concurrency']
        ),
        f"{row['pages_per_hour']:.2f} pph",
        f"{row['speedup_vs_bf16']:.4f}x",
        f"{row['mean_page_seconds']:.3f} s/page",
    )

print('Result directory:', VERIFY_OUTPUT)
print('Downloadable archive:', archive_path)
print('=' * 100)

In [ ]:
from IPython.display import FileLink, display
from pathlib import Path
import json
import shutil
import zipfile

# Correct logical-output validation.
checks[
    'all_runs_saved_27_markdown_files'
] = all(
    run['saved_markdown_files'] == 27
    and run['actual_markdown_files'] >= 27
    and len(run['measured_pages']) == 27
    and len({
        row['manifest_index']
        for row in run['measured_pages']
    }) == 27
    for run in verification_runs
)

verification_status = (
    'PASS'
    if all(checks.values())
    else 'REVIEW'
)

verification_report['status'] = (
    verification_status
)
verification_report['checks'] = checks
verification_report[
    'completed_utc'
] = datetime.now(
    timezone.utc
).isoformat()

summary_path = (
    VERIFY_OUTPUT
    / 'verification_summary.json'
)

summary_path.write_text(
    json.dumps(
        verification_report,
        indent=2,
    ) + '\n',
    encoding='utf-8',
)

# Create a shorter, easy-to-find archive name.
archive_base = Path(
    '/kaggle/working/'
    'l4_wna16_c1_c24_results'
)

download_archive = Path(
    str(archive_base) + '.zip'
)

# Preserve an existing archive instead of overwriting it.
if download_archive.exists():
    archive_base = Path(
        '/kaggle/working/'
        f'l4_wna16_c1_c24_results_{VERIFY_TAG}'
    )

download_archive = Path(
    shutil.make_archive(
        str(archive_base),
        'zip',
        root_dir=VERIFY_OUTPUT,
    )
)

assert download_archive.is_file()
assert download_archive.stat().st_size > 0

with zipfile.ZipFile(
    download_archive,
    'r',
) as archive:
    corrupt_member = archive.testzip()
    archive_members = archive.namelist()

assert corrupt_member is None
assert 'verification_summary.json' in (
    archive_members
)

print('=' * 100)
print(
    'VERIFICATION STATUS:',
    verification_status,
)
print(
    'Archive:',
    download_archive,
)
print(
    'Archive size MiB:',
    round(
        download_archive.stat().st_size
        / (1024 ** 2),
        2,
    ),
)
print(
    'Files inside archive:',
    len(archive_members),
)
print('=' * 100)

print('ZIP files currently in /kaggle/working:')

for path in sorted(
    Path('/kaggle/working').glob('*.zip')
):
    print(
        path.name,
        round(
            path.stat().st_size
            / (1024 ** 2),
            2,
        ),
        'MiB',
    )

display(
    FileLink(
        str(download_archive)
    )
)